# 🤖 FinSight — Complete AI Financial Chatbot

> **An end-to-end AI-powered financial assistant** for Indian stock markets, built with LangGraph, FastAPI, and Groq.

This notebook contains the **entire project source code** organized in the order it should be read and understood.

---

## 📋 Table of Contents

1. **Setup & Dependencies**
2. **Configuration** — Settings & Prompts
3. **Data Models** — Pydantic Schemas
4. **Utilities** — Caching Layer
5. **Data Services** — Market Data, News, Screener.in, Wikipedia
6. **Symbol Utilities** — Company Name → Ticker Resolution
7. **Trading Knowledge Base** — Built-in Educational Content
8. **Response Formatting** — Emoji-rich Output
9. **RAG Pipeline** — FAISS Vector Store + Wikipedia
10. **LangGraph Agent** — The Brain (ReAct Agent with 9 Tools)
11. **FastAPI Server** — REST, WebSocket, SSE Streaming
12. **Streamlit Frontend** — Chat UI
13. **Stock Screener** — Technical + Fundamental Analysis Engine
14. **Live Demo** — Test the Chatbot

---

## 🏗️ Architecture

```
User Message → FastAPI Server → LangGraph ReAct Agent → LLM picks tool(s)
                                     ↓
    ┌──────────────────────────────────────────────────────────┐
    │  9 Tools: Price, Details, History, News, Screen, Analyze,│
    │           Knowledge Base, Market Summary, Index Data      │
    └──────────────────────────────────────────────────────────┘
                                     ↓
    Data Services (yfinance, Screener.in, RSS, Wikipedia, FAISS)
                                     ↓
    LLM synthesizes final response → ChatResponse → User
```

**Tech Stack:** LangGraph • LangChain • Groq (LLaMA 3.1 70B) • FastAPI • yfinance • FAISS • Streamlit

---
# 1. Setup & Dependencies

Install all required packages. **Uncomment and run the cell below** if this is your first time.

In [ ]:
# ============================================================
# INSTALL ALL DEPENDENCIES (uncomment to run)
# ============================================================

# !pip install fastapi uvicorn pydantic pydantic-settings python-dotenv
# !pip install langchain langchain-groq langchain-community langchain-huggingface
# !pip install langgraph langgraph-checkpoint-sqlite
# !pip install yfinance feedparser aiohttp
# !pip install sentence-transformers faiss-cpu
# !pip install rapidfuzz wikipedia-api pandas-ta
# !pip install slowapi sentry-sdk[fastapi]
# !pip install streamlit

In [ ]:
# Common imports used throughout the project
import os
import json
import logging
import asyncio
import re
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any
from pathlib import Path

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('FinSight')
logger.info('FinSight notebook initialized')

---
# 2. Configuration

## 2.1 Environment Setup

The `.env` file stores API keys securely. Create one in your project root:
```
GROQ_API_KEY=your_groq_api_key_here
```

## 2.2 Settings (`common/config/settings.py`)

Centralized configuration using **Pydantic Settings**. All values are loaded from environment variables with smart defaults.

**Key concepts:** `BaseSettings`, `@lru_cache` singleton, `Field` with env binding.

In [ ]:
"""
Centralized configuration management for the Finance Chatbot.
Uses Pydantic Settings for environment variable support.
"""

from pydantic_settings import BaseSettings
from pydantic import Field
from typing import Optional
from functools import lru_cache


class Settings(BaseSettings):
    """Application settings loaded from environment variables."""
    
    # API Keys
    groq_api_key: str = Field(..., env="GROQ_API_KEY")
    news_api_key: Optional[str] = Field(None, env="NEWS_API_KEY")
    alpha_vantage_key: Optional[str] = Field(None, env="ALPHA_VANTAGE_KEY")
    
    # LLM Configuration
    llm_model: str = Field("llama-3.3-70b-versatile", env="LLM_MODEL")
    llm_temperature: float = Field(0.3, env="LLM_TEMPERATURE")
    llm_max_tokens: int = Field(2048, env="LLM_MAX_TOKENS")
    llm_model_fallback: str = Field("llama-3.3-70b-versatile", env="LLM_MODEL_FALLBACK")
    gemini_api_key: Optional[str] = Field(None, env="GEMINI_API_KEY")
    gemini_model: str = Field("gemini-1.5-flash", env="GEMINI_MODEL")
    
    # Embedding Configuration
    embedding_model: str = Field(
        "sentence-transformers/all-MiniLM-L6-v2", 
        env="EMBEDDING_MODEL"
    )
    
    # Cache Configuration
    cache_ttl_market_data: int = Field(30, env="CACHE_TTL_MARKET")  # seconds
    cache_ttl_news: int = Field(300, env="CACHE_TTL_NEWS")  # 5 minutes
    cache_ttl_session: int = Field(3600, env="CACHE_TTL_SESSION")  # 1 hour
    
    # Server Configuration
    server_host: str = Field("0.0.0.0", env="SERVER_HOST")
    server_port: int = Field(8000, env="SERVER_PORT")
    sentry_dsn: Optional[str] = Field(None, env="SENTRY_DSN")
    cors_origins: list[str] = Field(
        ["http://localhost:3000", "http://127.0.0.1:3000"],
        env="CORS_ORIGINS"
    )
    rate_limit_per_minute: int = Field(20, env="RATE_LIMIT_PER_MINUTE")
    rate_limit_enabled: bool = Field(True, env="RATE_LIMIT_ENABLED")
    
    # Feature Flags
    enable_streaming: bool = Field(True, env="ENABLE_STREAMING")
    enable_websocket: bool = Field(True, env="ENABLE_WEBSOCKET")
    enable_news: bool = Field(True, env="ENABLE_NEWS")
    enable_portfolio: bool = Field(True, env="ENABLE_PORTFOLIO")
    
    # Market Configuration
    default_market: str = Field("NSE", env="DEFAULT_MARKET")  # NSE, BSE, US
    
    # RAG Configuration
    rag_chunk_size: int = Field(500, env="RAG_CHUNK_SIZE")
    rag_chunk_overlap: int = Field(50, env="RAG_CHUNK_OVERLAP")
    rag_top_k: int = Field(3, env="RAG_TOP_K")
    
    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"
        extra = "ignore"


@lru_cache()
def get_settings() -> Settings:
    """Get cached settings instance."""
    return Settings()


# Convenience export
settings = get_settings()


## 2.3 Prompt Templates (`common/config/prompts.py`)

The chatbot's **personality** and **behavior rules**. The `AGENT_SYSTEM_PROMPT` tells the LangGraph agent which tools to use for which queries.

**Key concepts:** System prompt engineering, tool usage guidelines, disclaimer enforcement.

In [ ]:
"""
Centralized prompt templates for the Finance Chatbot.
All system prompts and templates are defined here for easy maintenance.
"""


# ============================================================================
# SYSTEM PROMPTS
# ============================================================================

FINSIGHT_PERSONA = """You are FinSight — an expert AI financial assistant for a modern trading platform.

Your core traits:
- Expert in Indian stock markets (NSE/BSE), mutual funds, and trading
- Clear, concise, and professional communication
- Use financial terminology correctly but explain complex terms
- Provide actionable insights, not just data
- Be helpful but NEVER give buy/sell/hold recommendations on specific stocks
- End EVERY stock-specific response with:
  ⚠️ Disclaimer: For informational purposes only, not investment advice. Consult a SEBI-registered advisor before investing.

Response guidelines:
- Keep responses focused and scannable
- Use bullet points for lists
- Include relevant emojis for visual appeal (📈 📉 💰 📊)
- Format numbers properly (₹1,234.56, +2.5%, 1.2Cr)
- Suggest follow-up actions when appropriate"""


# ============================================================================
# LANGGRAPH AGENT SYSTEM PROMPT
# ============================================================================

AGENT_SYSTEM_PROMPT = FINSIGHT_PERSONA + """

You are powered by a set of financial tools. Use them to answer queries accurately.

**Tool Usage Guidelines:**
- For stock prices: use get_stock_price with the stock symbol or company name
- For market overview: use get_market_summary
- For index data (Nifty, Sensex): use get_index_data
- For company info / PE / fundamentals: use get_stock_details
- For past performance / history: use get_stock_history (set days appropriately)
- For news: use get_stock_news (pass symbol or 'market' for general news)
- For educational concepts (what is X, explain Y): use search_knowledge_base
- For stock screening (undervalued, momentum): use screen_stocks
- For full stock analysis: use analyze_stock
- For greetings (hi, hello, bye, thanks): respond directly without tools
- For comparisons: call get_stock_details or get_stock_price for EACH stock being compared

**Response Formatting:**
- Use emojis for visual appeal (📈 📉 💰 📊 📰)
- Format Indian prices in ₹ with commas, US prices in $
- Keep responses concise and scannable with bullet points
**Presenting Tool Results (CRITICAL):**
- You MUST include the actual financial data returned by the tools in your final response!
- Do NOT assume the user has seen the tool output. You are the ONLY one who can show it to them.
- Structure your response exactly like this:
  1. The financial data (price, news, etc.)
  2. Your brief analysis or insight
  3. The disclaimer (if applicable)

**Important:**
- For stock-specific responses, ALWAYS include this at the very end:
  "⚠️ Disclaimer: For informational purposes only, not investment advice. Consult a SEBI-registered advisor."
- NEVER fabricate stock prices or financial data. Only report what the tools return.
- If a tool fails or returns no data, tell the user honestly.
- For ambiguous queries, ask the user to clarify rather than guessing.
- Do NOT include follow-up suggestions in your response — they are generated separately.
"""


---
# 3. Data Models (`common/models/schemas.py`)

All data structures defined as **Pydantic models** — giving us automatic validation, type safety, and auto-generated API docs.

**Models:** `ChatRequest`, `ChatResponse`, `StockPrice`, `IndexData`, `StockDetails`, `StockHistory`, `NewsArticle`, `StockAnalysis`, `TechnicalIndicators`, `FundamentalData`

In [ ]:
"""
Pydantic models/schemas for the Finance Chatbot.
Defines request/response structures and domain objects.
"""

from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from datetime import datetime
from enum import Enum


# ============================================================================
# ENUMS
# ============================================================================



class Market(str, Enum):
    """Supported stock markets."""
    NSE = "NSE"
    BSE = "BSE"
    US = "US"


# ============================================================================
# CHAT MODELS
# ============================================================================

class ChatMessage(BaseModel):
    """A single chat message."""
    role: str = Field(..., description="Role: 'user' or 'assistant'")
    content: str = Field(..., description="Message content")
    timestamp: datetime = Field(default_factory=datetime.now)
    metadata: Optional[Dict[str, Any]] = None


class ChatRequest(BaseModel):
    """Incoming chat request."""
    message: str = Field(..., min_length=1, max_length=2000)
    session_id: Optional[str] = Field(None, description="Session ID for context")
    include_context: bool = Field(True, description="Include conversation history")


class ChatResponse(BaseModel):
    """Chat response with metadata."""
    reply: str = Field(..., description="Bot response")
    intent: Optional[str] = Field(None, description="Detected intent")
    entities: Optional[Dict[str, Any]] = Field(None, description="Extracted entities")
    sources: Optional[List[str]] = Field(None, description="Source references")
    suggestions: Optional[List[str]] = Field(None, description="Follow-up suggestions")
    session_id: Optional[str] = Field(None, description="Session ID")


# ============================================================================
# MARKET DATA MODELS
# ============================================================================

class StockPrice(BaseModel):
    """Stock price data."""
    symbol: str
    name: Optional[str] = None
    price: float
    change: float = Field(..., description="Price change")
    change_percent: float = Field(..., description="Percentage change")
    volume: Optional[int] = None
    high: Optional[float] = None
    low: Optional[float] = None
    open: Optional[float] = None
    prev_close: Optional[float] = None
    timestamp: datetime = Field(default_factory=datetime.now)
    market: Market = Market.NSE


class IndexData(BaseModel):
    """Market index data."""
    symbol: str
    name: str
    value: float
    change: float
    change_percent: float
    timestamp: datetime = Field(default_factory=datetime.now)


class MarketMovers(BaseModel):
    """Top gainers and losers."""
    gainers: List[StockPrice]
    losers: List[StockPrice]
    timestamp: datetime = Field(default_factory=datetime.now)


class StockDetails(BaseModel):
    """Detailed stock information."""
    symbol: str
    name: str
    sector: Optional[str] = None
    industry: Optional[str] = None
    market_cap: Optional[float] = None
    pe_ratio: Optional[float] = None
    eps: Optional[float] = None
    dividend_yield: Optional[float] = None
    week_52_high: Optional[float] = None
    week_52_low: Optional[float] = None
    description: Optional[str] = None


class StockHistoryDay(BaseModel):
    """Single day of stock history."""
    date: str
    open: float
    high: float
    low: float
    close: float
    volume: Optional[int] = None
    change_percent: Optional[float] = None


class StockHistory(BaseModel):
    """Historical stock data over a period."""
    symbol: str
    name: Optional[str] = None
    days: List[StockHistoryDay]
    period: Optional[str] = "5d"
    overall_change_percent: Optional[float] = None
    market: Market = Market.NSE


# ============================================================================
# NEWS MODELS
# ============================================================================

class NewsArticle(BaseModel):
    """Financial news article."""
    title: str
    summary: Optional[str] = None
    source: str
    url: str
    published_at: datetime
    related_symbols: Optional[List[str]] = None
    sentiment: Optional[str] = None  # positive, negative, neutral


class NewsFeed(BaseModel):
    """Collection of news articles."""
    articles: List[NewsArticle]
    query: Optional[str] = None
    timestamp: datetime = Field(default_factory=datetime.now)


# ============================================================================
# PORTFOLIO MODELS
# ============================================================================

class PortfolioHolding(BaseModel):
    """Single portfolio holding."""
    symbol: str
    name: Optional[str] = None
    quantity: int
    avg_buy_price: float
    current_price: float
    current_value: float
    pnl: float = Field(..., description="Profit/Loss")
    pnl_percent: float


class PortfolioSummary(BaseModel):
    """Portfolio overview."""
    total_invested: float
    current_value: float
    total_pnl: float
    total_pnl_percent: float
    holdings: List[PortfolioHolding]
    last_updated: datetime = Field(default_factory=datetime.now)




# ============================================================================
# SCREENER MODELS
# ============================================================================

class TechnicalIndicators(BaseModel):
    """Technical analysis indicator values."""
    rsi: Optional[float] = None
    sma_20: Optional[float] = None
    sma_50: Optional[float] = None
    sma_200: Optional[float] = None
    ema_12: Optional[float] = None
    ema_26: Optional[float] = None
    macd: Optional[float] = None
    macd_signal: Optional[float] = None
    macd_histogram: Optional[float] = None
    bollinger_upper: Optional[float] = None
    bollinger_lower: Optional[float] = None
    bollinger_position: Optional[float] = None
    volume_ratio: Optional[float] = None
    supertrend_signal: Optional[str] = None
    adx: Optional[float] = None
    vwap_position: Optional[str] = None
    stochastic_k: Optional[float] = None


class FundamentalData(BaseModel):
    """Fundamental analysis data."""
    pe_ratio: Optional[float] = None
    pb_ratio: Optional[float] = None
    eps: Optional[float] = None
    market_cap: Optional[float] = None
    book_value: Optional[float] = None
    dividend_yield: Optional[float] = None
    roe: Optional[float] = None
    debt_to_equity: Optional[float] = None
    earnings_growth: Optional[float] = None
    revenue_growth: Optional[float] = None
    current_ratio: Optional[float] = None
    quick_ratio: Optional[float] = None
    free_cash_flow: Optional[float] = None
    institutional_holding: Optional[float] = None
    peg_ratio: Optional[float] = None
    profit_margin: Optional[float] = None
    sector: Optional[str] = None
    industry: Optional[str] = None


class StockAnalysis(BaseModel):
    """Complete analysis of a single stock (technical + fundamental)."""
    symbol: str
    name: Optional[str] = None
    price: float
    change_percent: float = 0.0
    technical: TechnicalIndicators = Field(default_factory=TechnicalIndicators)
    fundamental: FundamentalData = Field(default_factory=FundamentalData)
    signal: str = "NEUTRAL"  # BUY, SELL, HOLD, NEUTRAL
    score: float = 50.0  # 0-100 composite score
    market: Market = Market.NSE


class ScreenerResult(BaseModel):
    """Results from a stock screening operation."""
    screen_name: str
    description: str = ""
    stocks: List[StockAnalysis] = Field(default_factory=list)
    total_scanned: int = 0
    timestamp: datetime = Field(default_factory=datetime.now)


# ============================================================================
# API MODELS
# ============================================================================

class HealthResponse(BaseModel):
    """Health check response."""
    status: str = "healthy"
    version: str = "2.0.0"
    timestamp: datetime = Field(default_factory=datetime.now)


class ErrorResponse(BaseModel):
    """Error response."""
    error: str
    detail: Optional[str] = None
    code: Optional[str] = None


---
# 4. Caching Layer (`common/utils/cache.py`)

In-memory cache with **TTL (Time-To-Live)** support to avoid hammering APIs.

**Key concepts:** `OrderedDict` for LRU eviction, `threading.Lock` for thread safety, decorator pattern `@cache_with_ttl`.

In [ ]:
"""
Cache utilities for the Finance Chatbot.
Provides in-memory caching with TTL support.
"""

from datetime import datetime, timedelta
from typing import Any, Optional, Callable, TypeVar
from functools import wraps
from collections import OrderedDict
from threading import Lock

T = TypeVar('T')


class TTLCache:
    """
    Simple in-memory cache with TTL (Time To Live) support.
    Thread-safe with LRU eviction.
    """
    
    def __init__(self, max_size: int = 1000, default_ttl: int = 60):
        """
        Initialize cache.
        
        Args:
            max_size: Maximum number of items
            default_ttl: Default TTL in seconds
        """
        self._cache: OrderedDict[str, tuple] = OrderedDict()
        self._max_size = max_size
        self._default_ttl = default_ttl
        self._lock = Lock()
    
    def get(self, key: str) -> Optional[Any]:
        """Get value from cache if not expired."""
        with self._lock:
            if key not in self._cache:
                return None
            
            value, expiry = self._cache[key]
            
            if datetime.now() > expiry:
                del self._cache[key]
                return None
            
            # Move to end (LRU)
            self._cache.move_to_end(key)
            return value
    
    def set(self, key: str, value: Any, ttl: int = None) -> None:
        """Set value in cache with TTL."""
        ttl = ttl or self._default_ttl
        expiry = datetime.now() + timedelta(seconds=ttl)
        
        with self._lock:
            self._cache[key] = (value, expiry)
            self._cache.move_to_end(key)
            
            # Evict oldest if over capacity
            while len(self._cache) > self._max_size:
                self._cache.popitem(last=False)
    
    def delete(self, key: str) -> bool:
        """Delete a key from cache."""
        with self._lock:
            if key in self._cache:
                del self._cache[key]
                return True
            return False
    
    def clear(self) -> None:
        """Clear all cache entries."""
        with self._lock:
            self._cache.clear()
    
    def cleanup_expired(self) -> int:
        """Remove all expired entries. Returns count removed."""
        with self._lock:
            now = datetime.now()
            expired = [k for k, (v, exp) in self._cache.items() if now > exp]
            for k in expired:
                del self._cache[k]
            return len(expired)
    
    def stats(self) -> dict:
        """Get cache statistics."""
        with self._lock:
            return {
                "size": len(self._cache),
                "max_size": self._max_size,
                "default_ttl": self._default_ttl,
            }


# Global cache instances
_caches: dict = {}

def get_cache(name: str = "default", max_size: int = 1000, ttl: int = 60) -> TTLCache:
    """Get or create a named cache instance."""
    if name not in _caches:
        _caches[name] = TTLCache(max_size=max_size, default_ttl=ttl)
    return _caches[name]


def cache_with_ttl(ttl: int = 60, cache_name: str = "default"):
    """
    Decorator to cache function results with TTL.
    
    Args:
        ttl: Time to live in seconds
        cache_name: Name of cache to use
    """
    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @wraps(func)
        def wrapper(*args, **kwargs) -> T:
            cache = get_cache(cache_name)
            
            # Create cache key from function name and arguments
            key = f"{func.__name__}:{str(args)}:{str(sorted(kwargs.items()))}"
            
            # Check cache
            cached = cache.get(key)
            if cached is not None:
                return cached
            
            # Call function and cache result
            result = func(*args, **kwargs)
            cache.set(key, result, ttl)
            return result
        
        return wrapper
    return decorator


def async_cache_with_ttl(ttl: int = 60, cache_name: str = "default"):
    """
    Decorator to cache async function results with TTL.
    
    Args:
        ttl: Time to live in seconds
        cache_name: Name of cache to use
    """
    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @wraps(func)
        async def wrapper(*args, **kwargs) -> T:
            cache = get_cache(cache_name)
            
            # Create cache key
            key = f"{func.__name__}:{str(args)}:{str(sorted(kwargs.items()))}"
            
            # Check cache
            cached = cache.get(key)
            if cached is not None:
                return cached
            
            # Call async function and cache result
            result = await func(*args, **kwargs)
            cache.set(key, result, ttl)
            return result
        
        return wrapper
    return decorator


---
# 5. Data Services

These are the **data-fetching layer** — they connect to external APIs and return structured Pydantic objects.

## 5.1 Market Data Service (`common/data_services/market_data.py`)

Real-time stock prices, index data, and historical OHLCV data via **yfinance**.

**Key features:**
- Smart market detection: tries NSE first → falls back to US
- jugaad-trader for real-time NSE data
- 30-second in-memory caching
- Simulation fallback for demo purposes

In [ ]:
"""
Market data service for real-time stock and index information.
Uses yfinance as the primary data source (free, reliable for Indian stocks).
"""

import asyncio
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any
from functools import lru_cache
import logging

try:
    import yfinance as yf
except ImportError:
    yf = None

try:
    from jugaad_trader.nse import NSELive
    _nse_live = NSELive()
    JUGAAD_AVAILABLE = True
except ImportError:
    _nse_live = None
    JUGAAD_AVAILABLE = False
    
from common.models.schemas import StockPrice, IndexData, StockDetails, MarketMovers, Market, StockHistory, StockHistoryDay
from common.utils.cache import cache_with_ttl

logger = logging.getLogger(__name__)


# ============================================================================
# SYMBOL MAPPINGS
# ============================================================================

# NSE stock symbols need .NS suffix for yfinance
NSE_SUFFIX = ".NS"
BSE_SUFFIX = ".BO"

# Index mappings
INDEX_SYMBOLS = {
    "NIFTY 50": "^NSEI",
    "NIFTY50": "^NSEI",
    "NIFTY": "^NSEI",
    "SENSEX": "^BSESN",
    "BSE SENSEX": "^BSESN",
    "BANK NIFTY": "^NSEBANK",
    "BANKNIFTY": "^NSEBANK",
    "NIFTY IT": "^CNXIT",
    "NIFTYIT": "^CNXIT",
    "NIFTY BANK": "^NSEBANK",
}

# Popular stock name mappings
STOCK_NAME_MAP = {
    "RELIANCE": "Reliance Industries Ltd",
    "TCS": "Tata Consultancy Services",
    "HDFCBANK": "HDFC Bank Ltd",
    "INFY": "Infosys Ltd",
    "ICICIBANK": "ICICI Bank Ltd",
    "HINDUNILVR": "Hindustan Unilever Ltd",
    "ITC": "ITC Ltd",
    "SBIN": "State Bank of India",
    "BHARTIARTL": "Bharti Airtel Ltd",
    "KOTAKBANK": "Kotak Mahindra Bank Ltd",
    "LIC": "Life Insurance Corporation",
    "BAJFINANCE": "Bajaj Finance Ltd",
    "TATAMOTORS": "Tata Motors Ltd",
    "MARUTI": "Maruti Suzuki India Ltd",
    "ASIANPAINT": "Asian Paints Ltd",
    "WIPRO": "Wipro Ltd",
    "TITAN": "Titan Company Ltd",
    "ADANIENT": "Adani Enterprises Ltd",
    "AXISBANK": "Axis Bank Ltd",
    "SUNPHARMA": "Sun Pharmaceutical Industries Ltd",
}


class MarketDataService:
    """
    Service for fetching real-time and historical market data.
    Uses yfinance for data retrieval with caching.
    """
    
    def __init__(self, default_market: str = "NSE"):
        """
        Initialize market data service.
        
        Args:
            default_market: Default market (NSE or BSE)
        """
        self.default_market = default_market
        self._cache: Dict[str, tuple] = {}  # (value, expiry_time)
        self._cache_ttl = 30  # seconds
        
        if yf is None:
            logger.error("CRITICAL: yfinance not installed. Run: pip install yfinance")
    
    def _get_yf_symbol(self, symbol: str, market: str = None) -> str:
        """Convert symbol to yfinance format."""
        market = market or self.default_market
        
        # Check if it's an index
        if symbol.upper() in INDEX_SYMBOLS:
            return INDEX_SYMBOLS[symbol.upper()]
        
        # Add market suffix
        symbol = symbol.upper()
        if market == "US":
            return symbol  # US stocks use bare symbol
        elif market == "NSE" and not symbol.endswith(NSE_SUFFIX):
            return f"{symbol}{NSE_SUFFIX}"
        elif market == "BSE" and not symbol.endswith(BSE_SUFFIX):
            return f"{symbol}{BSE_SUFFIX}"
        
        return symbol
    
    def _detect_market(self, symbol: str) -> tuple:
        """
        Smart market detection: tries NSE first, then US.
        Returns (yf_symbol, detected_market) tuple.
        """
        if yf is None:
            return f"{symbol.upper()}{NSE_SUFFIX}", "NSE"
        
        symbol_upper = symbol.upper()
        
        # Check if it's an index
        if symbol_upper in INDEX_SYMBOLS:
            return INDEX_SYMBOLS[symbol_upper], "INDEX"
        
        # Try NSE first
        nse_symbol = f"{symbol_upper}{NSE_SUFFIX}"
        try:
            ticker = yf.Ticker(nse_symbol)
            info = ticker.info
            price = info.get('currentPrice') or info.get('regularMarketPrice', 0)
            if price and price > 0:
                return nse_symbol, "NSE"
        except Exception:
            pass
        
        # Try US market (bare symbol)
        try:
            ticker = yf.Ticker(symbol_upper)
            info = ticker.info
            price = info.get('currentPrice') or info.get('regularMarketPrice', 0)
            if price and price > 0:
                return symbol_upper, "US"
        except Exception:
            pass
        
        # Default to NSE
        return nse_symbol, "NSE"
    
    def _get_from_cache(self, key: str) -> Optional[Any]:
        """Get value from cache if not expired."""
        if key in self._cache:
            value, expiry = self._cache[key]
            if datetime.now() < expiry:
                return value
            del self._cache[key]
        return None
    
    def _set_cache(self, key: str, value: Any, ttl: int = None):
        """Set value in cache with TTL."""
        ttl = ttl or self._cache_ttl
        expiry = datetime.now() + timedelta(seconds=ttl)
        self._cache[key] = (value, expiry)
    
    async def get_stock_price(
        self, 
        symbol: str, 
        market: str = None
    ) -> Optional[StockPrice]:
        """
        Get current stock price. Automatically detects market if not specified.
        
        Args:
            symbol: Stock symbol (e.g., 'TCS', 'RELIANCE', 'NVDA', 'AAPL')
            market: Market hint (NSE, BSE, US) — auto-detected if None
            
        Returns:
            StockPrice object or None if not found
        """
        cache_key = f"price:{symbol}:{market or 'auto'}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        try:
            # Try jugaad-trader first for NSE stocks (more real-time)
            if JUGAAD_AVAILABLE and market != "US":
                try:
                    quote = _nse_live.get_quote(symbol.upper())
                    if quote and "priceInfo" in quote:
                        pi = quote["priceInfo"]
                        current = pi.get("lastPrice", 0)
                        prev = pi.get("previousClose", current)
                        if current and current > 0:
                            change = current - prev
                            change_pct = (change / prev * 100) if prev else 0
                            stock_price = StockPrice(
                                symbol=symbol.upper(),
                                name=STOCK_NAME_MAP.get(symbol.upper(), symbol),
                                price=round(current, 2),
                                change=round(change, 2),
                                change_percent=round(change_pct, 2),
                                volume=pi.get("totalTradedVolume"),
                                high=pi.get("intraDayHighLow", {}).get("max"),
                                low=pi.get("intraDayHighLow", {}).get("min"),
                                prev_close=prev,
                                market=Market.NSE
                            )
                            self._set_cache(cache_key, stock_price)
                            return stock_price
                except Exception as je:
                    logger.debug(f"jugaad-trader failed for {symbol}, falling back to yfinance: {je}")

            if yf is None:
                return None
            
            # Smart detection: try NSE first, then US
            if market:
                yf_symbol = self._get_yf_symbol(symbol, market)
                detected_market = market
            else:
                yf_symbol, detected_market = self._detect_market(symbol)
            
            # Fetch data
            ticker = yf.Ticker(yf_symbol)
            info = ticker.info
            
            # Get current price data
            current_price = info.get('currentPrice') or info.get('regularMarketPrice', 0)
            if not current_price or current_price == 0:
                return None
            
            prev_close = info.get('previousClose', info.get('regularMarketPreviousClose', current_price))
            
            change = current_price - prev_close
            change_percent = (change / prev_close * 100) if prev_close else 0
            
            # Determine currency symbol for formatting
            currency = "$" if detected_market == "US" else "\u20b9"
            
            stock_price = StockPrice(
                symbol=symbol.upper(),
                name=STOCK_NAME_MAP.get(symbol.upper(), info.get('shortName', symbol)),
                price=round(current_price, 2),
                change=round(change, 2),
                change_percent=round(change_percent, 2),
                volume=info.get('volume', info.get('regularMarketVolume')),
                high=info.get('dayHigh', info.get('regularMarketDayHigh')),
                low=info.get('dayLow', info.get('regularMarketDayLow')),
                open=info.get('open', info.get('regularMarketOpen')),
                prev_close=prev_close,
                market=Market.US if detected_market == "US" else (
                    Market.BSE if detected_market == "BSE" else Market.NSE
                )
            )
            
            self._set_cache(cache_key, stock_price)
            return stock_price
            
        except Exception as e:
            logger.error(f"Error fetching price for {symbol}: {e}")
            return None
    
    async def get_index_data(self, index: str) -> Optional[IndexData]:
        """
        Get market index data.
        
        Args:
            index: Index name (e.g., 'NIFTY 50', 'SENSEX')
            
        Returns:
            IndexData object or None
        """
        cache_key = f"index:{index}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        try:
            yf_symbol = INDEX_SYMBOLS.get(index.upper())
            if not yf_symbol:
                logger.warning(f"Unknown index: {index}")
                return None
            
            if yf is None:
                return None
            
            ticker = yf.Ticker(yf_symbol)
            hist = ticker.history(period="2d")
            
            if hist.empty:
                return None
            
            current = hist['Close'].iloc[-1]
            prev = hist['Close'].iloc[-2] if len(hist) > 1 else current
            change = current - prev
            change_percent = (change / prev * 100) if prev else 0
            
            index_data = IndexData(
                symbol=yf_symbol,
                name=index.upper(),
                value=round(current, 2),
                change=round(change, 2),
                change_percent=round(change_percent, 2)
            )
            
            self._set_cache(cache_key, index_data)
            return index_data
            
        except Exception as e:
            logger.error(f"Error fetching index {index}: {e}")
            return None
    
    async def get_stock_details(self, symbol: str) -> Optional[StockDetails]:
        """
        Get detailed stock information.
        
        Args:
            symbol: Stock symbol
            
        Returns:
            StockDetails object or None
        """
        cache_key = f"details:{symbol}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        try:
            yf_symbol = self._get_yf_symbol(symbol)
            
            if yf is None:
                return None
            
            ticker = yf.Ticker(yf_symbol)
            info = ticker.info
            
            details = StockDetails(
                symbol=symbol.upper(),
                name=info.get('shortName', symbol),
                sector=info.get('sector'),
                industry=info.get('industry'),
                market_cap=info.get('marketCap'),
                pe_ratio=info.get('trailingPE'),
                eps=info.get('trailingEps'),
                dividend_yield=info.get('dividendYield'),
                week_52_high=info.get('fiftyTwoWeekHigh'),
                week_52_low=info.get('fiftyTwoWeekLow'),
                description=info.get('longBusinessSummary')
            )
            
            # Cache for longer (5 minutes) as this data changes less frequently
            self._set_cache(cache_key, details, ttl=300)
            return details
            
        except Exception as e:
            logger.error(f"Error fetching details for {symbol}: {e}")
            return None
    
    async def get_market_summary(self) -> Dict[str, Any]:
        """Get overall market summary with key indices."""
        indices = ["NIFTY 50", "SENSEX", "BANK NIFTY"]
        results = {}
        
        for index in indices:
            data = await self.get_index_data(index)
            if data:
                results[index] = data
        
        return results
    
    async def get_stock_history(
        self,
        symbol: str,
        days: int = 5,
        market: str = None
    ) -> Optional[StockHistory]:
        """
        Get historical stock data for last N days.
        
        Args:
            symbol: Stock symbol (e.g., 'TCS', 'NVDA')
            days: Number of days of history (default 5)
            market: Market hint (auto-detected if None)
            
        Returns:
            StockHistory object or None
        """
        cache_key = f"history:{symbol}:{days}:{market or 'auto'}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        try:
            if yf is None:
                return None
            
            # Smart detection
            if market:
                yf_symbol = self._get_yf_symbol(symbol, market)
                detected_market = market
            else:
                yf_symbol, detected_market = self._detect_market(symbol)
            
            ticker = yf.Ticker(yf_symbol)
            # Fetch extra days to ensure we get enough trading days
            period = f"{days + 5}d"
            hist = ticker.history(period=period)
            
            if hist.empty:
                return None
            
            # Take last N rows
            hist = hist.tail(days)
            
            history_days = []
            prev_close = None
            for date, row in hist.iterrows():
                change_pct = None
                if prev_close and prev_close > 0:
                    change_pct = round((row['Close'] - prev_close) / prev_close * 100, 2)
                
                history_days.append(StockHistoryDay(
                    date=date.strftime('%Y-%m-%d'),
                    open=round(row['Open'], 2),
                    high=round(row['High'], 2),
                    low=round(row['Low'], 2),
                    close=round(row['Close'], 2),
                    volume=int(row['Volume']) if row['Volume'] else None,
                    change_percent=change_pct
                ))
                prev_close = row['Close']
            
            # Calculate overall change
            overall_change = None
            if len(history_days) >= 2:
                first_close = hist['Close'].iloc[0]
                last_close = hist['Close'].iloc[-1]
                if first_close > 0:
                    overall_change = round((last_close - first_close) / first_close * 100, 2)
            
            stock_name = STOCK_NAME_MAP.get(symbol.upper(), 
                                            ticker.info.get('shortName', symbol.upper()))
            
            history = StockHistory(
                symbol=symbol.upper(),
                name=stock_name,
                days=history_days,
                period=f"{days}d",
                overall_change_percent=overall_change,
                market=Market.US if detected_market == "US" else Market.NSE
            )
            
            self._set_cache(cache_key, history, ttl=60)  # Cache for 1 minute
            return history
            
        except Exception as e:
            logger.error(f"Error fetching history for {symbol}: {e}")
            return self._simulate_stock_history(symbol, days)
    
    # ========================================================================
    # SIMULATION METHODS (fallback when yfinance unavailable)
    # ========================================================================
    
    def _simulate_stock_price(self, symbol: str) -> StockPrice:
        """Generate simulated stock price for demo purposes."""
        import random
        base_price = random.uniform(100, 5000)
        change = random.uniform(-50, 50)
        
        return StockPrice(
            symbol=symbol.upper(),
            name=STOCK_NAME_MAP.get(symbol.upper(), f"{symbol} Ltd"),
            price=round(base_price, 2),
            change=round(change, 2),
            change_percent=round(change / base_price * 100, 2),
            volume=random.randint(100000, 10000000),
            high=round(base_price * 1.02, 2),
            low=round(base_price * 0.98, 2),
            market=Market.NSE
        )
    
    def _simulate_index_data(self, index: str) -> IndexData:
        """Generate simulated index data for demo purposes."""
        import random
        base_values = {
            "NIFTY 50": 22500,
            "SENSEX": 74000,
            "BANK NIFTY": 48000,
        }
        base = base_values.get(index.upper(), 20000)
        change = random.uniform(-200, 200)
        
        return IndexData(
            symbol=INDEX_SYMBOLS.get(index.upper(), index),
            name=index.upper(),
            value=round(base + random.uniform(-100, 100), 2),
            change=round(change, 2),
            change_percent=round(change / base * 100, 2)
        )
    
    def _simulate_stock_details(self, symbol: str) -> StockDetails:
        """Generate simulated stock details for demo purposes."""
        import random
        return StockDetails(
            symbol=symbol.upper(),
            name=STOCK_NAME_MAP.get(symbol.upper(), f"{symbol} Ltd"),
            sector="Technology",
            industry="IT Services",
            market_cap=random.randint(10000, 1000000) * 10000000,
            pe_ratio=round(random.uniform(10, 50), 2),
            eps=round(random.uniform(10, 200), 2),
            week_52_high=round(random.uniform(1000, 5000), 2),
            week_52_low=round(random.uniform(500, 2000), 2),
        )
    
    def _simulate_stock_history(self, symbol: str, days: int = 5) -> StockHistory:
        """Generate simulated stock history for demo purposes."""
        import random
        from datetime import date
        
        base_price = random.uniform(100, 5000)
        history_days = []
        
        for i in range(days):
            day_date = date.today() - timedelta(days=days - i)
            change = random.uniform(-3, 3)
            close = round(base_price * (1 + change / 100), 2)
            
            history_days.append(StockHistoryDay(
                date=day_date.strftime('%Y-%m-%d'),
                open=round(close * random.uniform(0.99, 1.01), 2),
                high=round(close * random.uniform(1.0, 1.03), 2),
                low=round(close * random.uniform(0.97, 1.0), 2),
                close=close,
                volume=random.randint(100000, 10000000),
                change_percent=round(change, 2) if i > 0 else None
            ))
            base_price = close
        
        overall = None
        if len(history_days) >= 2:
            first = history_days[0].close
            last = history_days[-1].close
            overall = round((last - first) / first * 100, 2)
        
        return StockHistory(
            symbol=symbol.upper(),
            name=STOCK_NAME_MAP.get(symbol.upper(), f"{symbol} Ltd"),
            days=history_days,
            period=f"{days}d",
            overall_change_percent=overall,
            market=Market.NSE
        )


# Singleton instance
_market_service = None

def get_market_data_service() -> MarketDataService:
    """Get or create the market data service singleton."""
    global _market_service
    if _market_service is None:
        _market_service = MarketDataService()
    return _market_service


## 5.2 News Service (`common/data_services/news_service.py`)

Financial news from **Google News RSS feeds** — no API key required!

**Key features:**
- Stock-specific and general market news
- HTML tag stripping, date parsing
- 5-minute caching, demo news fallback

In [ ]:
"""
News service for fetching and summarizing financial news.
Uses multiple sources: NewsAPI, RSS feeds, and web scraping fallback.
"""

import asyncio
from datetime import datetime, timedelta
from typing import List, Optional, Dict, Any
import logging
import re

try:
    import feedparser
except ImportError:
    feedparser = None

try:
    import aiohttp
except ImportError:
    aiohttp = None

from common.models.schemas import NewsArticle, NewsFeed

logger = logging.getLogger(__name__)


# ============================================================================
# NEWS SOURCE CONFIGURATION
# ============================================================================

# Google Finance RSS feeds (free, no API key needed)
RSS_FEEDS = {
    "market": "https://news.google.com/rss/search?q=indian+stock+market&hl=en-IN&gl=IN&ceid=IN:en",
    "nse": "https://news.google.com/rss/search?q=NSE+stocks&hl=en-IN&gl=IN&ceid=IN:en",
    "business": "https://news.google.com/rss/search?q=business+india&hl=en-IN&gl=IN&ceid=IN:en",
    "economy": "https://news.google.com/rss/search?q=indian+economy&hl=en-IN&gl=IN&ceid=IN:en",
}

# Stock-specific news search template
STOCK_NEWS_URL = "https://news.google.com/rss/search?q={symbol}+stock+india&hl=en-IN&gl=IN&ceid=IN:en"


class NewsService:
    """
    Service for fetching and processing financial news.
    Supports RSS feeds and optional NewsAPI integration.
    """
    
    def __init__(self, news_api_key: Optional[str] = None):
        """
        Initialize news service.
        
        Args:
            news_api_key: Optional NewsAPI.org API key
        """
        self.news_api_key = news_api_key
        self._cache: Dict[str, tuple] = {}
        self._cache_ttl = 300  # 5 minutes
        
        if feedparser is None:
            logger.warning("feedparser not installed. News features may be limited.")
    
    def _get_from_cache(self, key: str) -> Optional[Any]:
        """Get value from cache if not expired."""
        if key in self._cache:
            value, expiry = self._cache[key]
            if datetime.now() < expiry:
                return value
            del self._cache[key]
        return None
    
    def _set_cache(self, key: str, value: Any):
        """Set value in cache."""
        expiry = datetime.now() + timedelta(seconds=self._cache_ttl)
        self._cache[key] = (value, expiry)
    
    async def get_market_news(self, limit: int = 5) -> List[NewsArticle]:
        """
        Get latest market news.
        
        Args:
            limit: Maximum number of articles
            
        Returns:
            List of NewsArticle objects
        """
        cache_key = f"news:market:{limit}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        articles = await self._fetch_rss_news(RSS_FEEDS["market"], limit)
        
        if not articles:
            articles = self._get_fallback_news()
        
        self._set_cache(cache_key, articles)
        return articles
    
    async def get_stock_news(self, symbol: str, limit: int = 5) -> List[NewsArticle]:
        """
        Get news for a specific stock.
        
        Args:
            symbol: Stock symbol
            limit: Maximum number of articles
            
        Returns:
            List of NewsArticle objects
        """
        cache_key = f"news:stock:{symbol}:{limit}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached
        
        url = STOCK_NEWS_URL.format(symbol=symbol)
        articles = await self._fetch_rss_news(url, limit)
        
        # Filter to ensure articles are actually related to the stock
        filtered = [
            a for a in articles 
            if symbol.lower() in a.title.lower() or 
               symbol.lower() in (a.summary or "").lower()
        ]
        
        result = filtered if filtered else articles[:limit]
        self._set_cache(cache_key, result)
        return result
    
    async def _fetch_rss_news(self, url: str, limit: int = 5) -> List[NewsArticle]:
        """
        Fetch news from RSS feed.
        
        Args:
            url: RSS feed URL
            limit: Maximum articles
            
        Returns:
            List of NewsArticle objects
        """
        if feedparser is None:
            return self._get_fallback_news()[:limit]
        
        try:
            # Use asyncio to not block
            loop = asyncio.get_event_loop()
            feed = await loop.run_in_executor(None, feedparser.parse, url)
            
            articles = []
            for entry in feed.entries[:limit]:
                # Parse published date
                published = datetime.now()
                if hasattr(entry, 'published_parsed') and entry.published_parsed:
                    try:
                        published = datetime(*entry.published_parsed[:6])
                    except:
                        pass
                
                # Clean up title (remove source suffix)
                title = entry.title
                if " - " in title:
                    title = title.rsplit(" - ", 1)[0]
                
                # Get summary
                summary = entry.get('summary', '')
                # Remove HTML tags
                summary = re.sub(r'<[^>]+>', '', summary)[:200]
                
                articles.append(NewsArticle(
                    title=title,
                    summary=summary if summary else None,
                    source=entry.get('source', {}).get('title', 'Google News'),
                    url=entry.link,
                    published_at=published
                ))
            
            return articles
            
        except Exception as e:
            logger.error(f"Error fetching RSS news: {e}")
            return self._get_fallback_news()[:limit]
    
    def _get_fallback_news(self) -> List[NewsArticle]:
        """Return demo news when actual news unavailable."""
        now = datetime.now()
        return [
            NewsArticle(
                title="Markets show mixed trends amid global uncertainty",
                summary="Indian markets traded flat today with Nifty hovering around key levels.",
                source="FinSight Demo",
                url="https://example.com/news/1",
                published_at=now
            ),
            NewsArticle(
                title="Banking stocks lead gains in early trade",
                summary="HDFC Bank and ICICI Bank among top gainers as sector shows strength.",
                source="FinSight Demo",
                url="https://example.com/news/2",
                published_at=now - timedelta(hours=1)
            ),
            NewsArticle(
                title="IT sector faces headwinds on global tech slowdown concerns",
                summary="TCS, Infosys trade lower amid concerns over US recession.",
                source="FinSight Demo",
                url="https://example.com/news/3",
                published_at=now - timedelta(hours=2)
            ),
            NewsArticle(
                title="Auto sales data exceeds expectations for December",
                summary="Maruti, Tata Motors report better-than-expected monthly sales.",
                source="FinSight Demo",
                url="https://example.com/news/4",
                published_at=now - timedelta(hours=3)
            ),
            NewsArticle(
                title="RBI policy decision awaited by markets",
                summary="Analysts expect status quo on interest rates in upcoming MPC meeting.",
                source="FinSight Demo",
                url="https://example.com/news/5",
                published_at=now - timedelta(hours=4)
            ),
        ]
    
    def format_news(self, articles: List[NewsArticle], include_summary: bool = True) -> str:
        """
        Format news articles into a conversational response.
        
        Args:
            articles: List of news articles
            include_summary: Whether to include article summaries
            
        Returns:
            Formatted news string
        """
        if not articles:
            return "📰 No news articles available at the moment. Please try again later."
        
        response = "📰 **Latest Financial News**\n\n"
        
        for i, article in enumerate(articles, 1):
            # Time ago
            time_ago = self._format_time_ago(article.published_at)
            
            response += f"**{i}. {article.title}**\n"
            response += f"   ⏰ {time_ago} • 📌 {article.source}\n"
            
            if include_summary and article.summary:
                response += f"   {article.summary[:150]}{'...' if len(article.summary) > 150 else ''}\n"
            
            response += "\n"
        
        return response.strip()
    
    def _format_time_ago(self, dt: datetime) -> str:
        """Format datetime as 'X hours ago' style."""
        now = datetime.now()
        diff = now - dt
        
        if diff.days > 0:
            return f"{diff.days}d ago"
        elif diff.seconds >= 3600:
            hours = diff.seconds // 3600
            return f"{hours}h ago"
        elif diff.seconds >= 60:
            minutes = diff.seconds // 60
            return f"{minutes}m ago"
        else:
            return "Just now"


# Singleton instance
_news_service = None

def get_news_service(api_key: Optional[str] = None) -> NewsService:
    """Get or create the news service singleton."""
    global _news_service
    if _news_service is None:
        _news_service = NewsService(api_key)
    return _news_service


## 5.3 Screener.in Service (`common/data_services/screener_in_service.py`)

10-year fundamental data for Indian stocks — **completely free, no API key**.

Returns: PE, PB, ROE, ROCE, debt/equity, promoter/FII/DII holdings, sector, pros/cons.

In [ ]:
"""
Screener.in free API service.
Provides 10-year fundamental data for Indian stocks.
No API key required. Free public endpoint.
URL: https://www.screener.in/api/company/{SYMBOL}/
"""

import logging
from typing import Optional, Dict, Any
from datetime import datetime, timedelta

logger = logging.getLogger(__name__)
SCREENER_API_BASE = "https://www.screener.in/api/company"


class ScreenerInService:

    def __init__(self):
        self._cache: Dict[str, tuple] = {}
        self._cache_ttl = 3600

    def _get_from_cache(self, key):
        if key in self._cache:
            value, expiry = self._cache[key]
            if datetime.now() < expiry:
                return value
            del self._cache[key]
        return None

    def _set_cache(self, key, value):
        self._cache[key] = (value, datetime.now() + timedelta(seconds=self._cache_ttl))

    async def get_fundamentals(self, symbol: str) -> Optional[Dict[str, Any]]:
        cache_key = f"screener:{symbol}"
        cached = self._get_from_cache(cache_key)
        if cached:
            return cached

        url = f"{SCREENER_API_BASE}/{symbol.upper()}/"
        try:
            import urllib.request, json
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as r:
                data = json.loads(r.read())
            result = self._parse(symbol, data)
            self._set_cache(cache_key, result)
            return result
        except Exception as e:
            logger.error(f"Screener.in fetch failed for {symbol}: {e}")
            return None

    def _parse(self, symbol: str, data: dict) -> Dict[str, Any]:
        ratios = {}
        for r in data.get("ratios", []):
            vals = r.get("values", [{}])
            ratios[r.get("name", "")] = vals[-1].get("value") if vals else None

        return {
            "symbol": symbol.upper(),
            "name": data.get("name", symbol),
            "pe_ratio": ratios.get("Stock P/E"),
            "pb_ratio": ratios.get("Price to Book value"),
            "roe": ratios.get("Return on equity"),
            "roce": ratios.get("Return on capital employed"),
            "debt_to_equity": ratios.get("Debt to equity"),
            "current_ratio": ratios.get("Current ratio"),
            "dividend_yield": ratios.get("Dividend Yield"),
            "promoter_holding": data.get("shareholding", {}).get("promoters"),
            "fii_holding": data.get("shareholding", {}).get("fii"),
            "dii_holding": data.get("shareholding", {}).get("dii"),
            "sector": data.get("sector"),
            "industry": data.get("industry"),
            "pros": data.get("pros", [])[:3],
            "cons": data.get("cons", [])[:3],
            "source": "screener.in",
        }

    def format_for_llm(self, data: Dict[str, Any]) -> str:
        if not data:
            return ""
        lines = [f"Screener.in data for {data.get('name', '')} ({data.get('symbol', '')}):"]
        for k, label in [
            ("pe_ratio","PE Ratio"), ("pb_ratio","PB Ratio"), ("roe","ROE %"),
            ("roce","ROCE %"), ("debt_to_equity","Debt/Equity"),
            ("promoter_holding","Promoter Holding %"),
            ("fii_holding","FII Holding %"), ("current_ratio","Current Ratio")
        ]:
            if data.get(k) is not None:
                lines.append(f"{label}: {data[k]}")
        if data.get("pros"):
            lines.append(f"Strengths: {', '.join(data['pros'])}")
        if data.get("cons"):
            lines.append(f"Concerns: {', '.join(data['cons'])}")
        return "\n".join(lines)


_service = None
def get_screener_in_service() -> ScreenerInService:
    global _service
    if _service is None:
        _service = ScreenerInService()
    return _service


## 5.4 Wikipedia Service (`common/data_services/wikipedia_service.py`)

Company summaries and financial concept definitions from **Wikipedia REST API**.

In [ ]:
"""
Wikipedia Service.
Provides a fallback knowledge base for financial concepts.
Uses the free Wikipedia API.
"""

import logging
from typing import Optional
from functools import lru_cache

logger = logging.getLogger(__name__)

try:
    import wikipediaapi
except ImportError:
    wikipediaapi = None
    logger.warning("wikipedia-api package not found. Wikipedia service disabled.")


class WikipediaService:
    """Service to fetch summaries of financial concepts from Wikipedia."""
    
    def __init__(self):
        if wikipediaapi:
            # Requires a meaningful user agent
            self.wiki = wikipediaapi.Wikipedia(
                user_agent='FinSightBot/1.0 (https://github.com/FinSight)',
                language='en',
                extract_format=wikipediaapi.ExtractFormat.WIKI
            )
        else:
            self.wiki = None

    @lru_cache(maxsize=128)
    def search_concept(self, query: str) -> Optional[str]:
        """
        Search Wikipedia for a concept and return its summary.
        Adds 'finance' or 'economics' context to help disambiguate.
        """
        if not self.wiki:
            return None
            
        search_terms = [
            f"{query} finance",
            f"{query} economics",
            f"{query} investment",
            query
        ]
        
        for term in search_terms:
            try:
                page = self.wiki.page(term)
                if page.exists():
                    # Check if the title is actually somewhat related to the query
                    # to prevent completely random pages if the exact match failed
                    title_lower = page.title.lower()
                    query_words = query.lower().split()
                    
                    # Ensure at least one word from the query is in the title,
                    # or it's a very short query
                    if any(w in title_lower for w in query_words) or len(query_words) <= 1:
                        summary = page.summary
                        if summary and len(summary) > 50:
                            # Return the first 2-3 paragraphs (approx 800 chars)
                            truncated = summary[:800]
                            if len(summary) > 800:
                                last_period = truncated.rfind('.')
                                if last_period > 0:
                                    truncated = truncated[:last_period + 1]
                            return truncated
            except Exception as e:
                logger.debug(f"Wikipedia search failed for '{term}': {e}")
                
        return None

    def format_for_llm(self, concept: str, summary: str) -> str:
        """Format the Wikipedia summary for the LLM context."""
        return f"WIKIPEDIA KNOWLEDGE - {concept}:\n{summary}"


_wiki_service = None

def get_wikipedia_service() -> WikipediaService:
    """Get the Wikipedia service singleton."""
    global _wiki_service
    if _wiki_service is None:
        _wiki_service = WikipediaService()
    return _wiki_service


---
# 6. Symbol Utilities (`chatbot/core/symbol_utils.py`)

Maps natural company names to stock tickers. This is the **NLP bridge** between human language and market APIs.

**Resolution chain:**
1. Exact match in `COMPANY_NAME_MAP` ("reliance" → RELIANCE)
2. Direct symbol lookup in `NSE_KNOWN_SYMBOLS`
3. **Fuzzy match** via rapidfuzz ("reliace" → RELIANCE, score ≥ 85%)
4. **Fallback:** difflib if rapidfuzz is not installed

In [ ]:
"""
Symbol resolution utilities for the FinSight chatbot.
Maps natural company names to stock tickers and normalizes symbols.
Extracted from entity_extractor.py for use by LangGraph agent tools.
"""

import re
import logging
from typing import Optional

logger = logging.getLogger(__name__)


# ============================================================================
# TOP NSE SYMBOL WHITELIST
# ============================================================================

NSE_KNOWN_SYMBOLS: set = {
    # Nifty 50 & large-caps
    "TCS", "RELIANCE", "HDFCBANK", "INFY", "ICICIBANK", "HINDUNILVR",
    "SBIN", "BAJFINANCE", "BHARTIARTL", "KOTAKBANK", "WIPRO", "HCLTECH",
    "ASIANPAINT", "AXISBANK", "MARUTI", "SUNPHARMA", "TITAN", "TATAMOTORS",
    "TATASTEEL", "ULTRACEMCO", "LT", "NTPC", "POWERGRID", "ONGC",
    "COALINDIA", "JSWSTEEL", "HINDALCO", "DRREDDY", "CIPLA", "DIVISLAB",
    "ADANIENT", "ADANIPORTS", "ADANIGREEN", "ADANIPOWER", "BAJAJ-AUTO",
    "BAJAJFINSV", "TECHM", "GRASIM", "NESTLEIND", "ITC", "BRITANNIA",
    "DABUR", "MARICO", "GODREJCP", "VEDL", "ZOMATO", "PAYTM", "IRCTC",
    "HAL", "BHEL", "LICI", "TATAPOWER", "TATACHEM", "TATACONSUM",
    "TATAELXSI", "HEROMOTOCO", "EICHERMOT", "ASHOKLEY", "INDUSINDBK",
    "BANKBARODA", "PNB", "YESBANK", "IDFCFIRSTB", "LTIM", "PERSISTENT",
    "COFORGE", "DMART", "M&M",
    # Mid-caps frequently asked about
    "TATVA", "IRFC", "NHPC", "RECLTD", "PFC", "SAIL",
    "NMDC", "RVNL", "IRCON", "HUDCO", "CANBK", "UNIONBANK",
    "FEDERALBNK", "BANDHANBNK", "CHOLAFIN", "MUTHOOTFIN",
    "SBICARD", "HDFCLIFE", "ICICIPRULI", "SBILIFE",
    "PGHL", "AUROPHARMA", "LUPIN", "TORNTPHARM", "ALKEM",
    "PIIND", "NAUKRI", "INDIGO", "SPICEJET", "TRENT",
    "VOLTAS", "HAVELLS", "POLYCAB", "DIXON", "ABB",
    "SIEMENS", "CUMMINSIND", "THERMAX", "GMRINFRA", "ADANIGAS",
    # US stocks commonly asked about
    "NVDA", "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA",
    "NFLX", "AMD", "INTC", "DIS", "WMT", "KO", "PEP", "JNJ",
    "JPM", "GS", "PYPL", "CRM", "ADBE", "UBER", "SPOT",
    "SNAP", "ABNB", "PLTR", "COIN", "SNOW", "CRWD", "SHOP",
    "BA", "GM", "V", "MA", "BRK-B",
}


# ============================================================================
# COMPANY NAME → TICKER MAP
# ============================================================================

COMPANY_NAME_MAP = {
    # ------ INDIAN STOCKS (NSE) ------
    # Tata Group
    "tata steel": "TATASTEEL", "tata motors": "TATAMOTORS",
    "tata power": "TATAPOWER", "tata chemicals": "TATACHEM",
    "tata consumer": "TATACONSUM", "tata elxsi": "TATAELXSI",
    "tcs": "TCS", "tata consultancy": "TCS",
    # Reliance
    "reliance": "RELIANCE", "reliance industries": "RELIANCE", "jio": "RELIANCE",
    # Banking
    "hdfc bank": "HDFCBANK", "hdfc": "HDFCBANK",
    "icici bank": "ICICIBANK", "icici": "ICICIBANK",
    "sbi": "SBIN", "state bank": "SBIN", "state bank of india": "SBIN",
    "kotak bank": "KOTAKBANK", "kotak mahindra": "KOTAKBANK",
    "axis bank": "AXISBANK", "indusind bank": "INDUSINDBK",
    "bank of baroda": "BANKBARODA", "punjab national bank": "PNB", "pnb": "PNB",
    "yes bank": "YESBANK", "idfc first bank": "IDFCFIRSTB",
    # IT
    "infosys": "INFY", "wipro": "WIPRO",
    "hcl tech": "HCLTECH", "hcl technologies": "HCLTECH",
    "tech mahindra": "TECHM", "ltimindtree": "LTIM",
    "persistent systems": "PERSISTENT", "coforge": "COFORGE",
    # Automobile
    "maruti": "MARUTI", "maruti suzuki": "MARUTI",
    "mahindra": "M&M", "mahindra and mahindra": "M&M", "m&m": "M&M",
    "bajaj auto": "BAJAJ-AUTO", "hero motocorp": "HEROMOTOCO",
    "eicher motors": "EICHERMOT", "ashok leyland": "ASHOKLEY",
    # Pharma
    "sun pharma": "SUNPHARMA", "dr reddy": "DRREDDY", "dr reddys": "DRREDDY",
    "cipla": "CIPLA", "divi's lab": "DIVISLAB", "divis lab": "DIVISLAB",
    # FMCG
    "hindustan unilever": "HINDUNILVR", "hul": "HINDUNILVR",
    "itc": "ITC", "nestle": "NESTLEIND", "nestle india": "NESTLEIND",
    "britannia": "BRITANNIA", "dabur": "DABUR",
    "godrej consumer": "GODREJCP", "marico": "MARICO",
    # Others
    "adani enterprises": "ADANIENT", "adani ports": "ADANIPORTS",
    "adani green": "ADANIGREEN", "adani power": "ADANIPOWER",
    "bajaj finance": "BAJFINANCE", "bajaj finserv": "BAJAJFINSV",
    "asian paints": "ASIANPAINT", "titan": "TITAN", "titan company": "TITAN",
    "ultratech cement": "ULTRACEMCO", "grasim": "GRASIM",
    "bharti airtel": "BHARTIARTL", "airtel": "BHARTIARTL",
    "lic": "LICI", "life insurance corporation": "LICI",
    "power grid": "POWERGRID", "ntpc": "NTPC", "ongc": "ONGC",
    "coal india": "COALINDIA", "hindalco": "HINDALCO",
    "jsw steel": "JSWSTEEL", "vedanta": "VEDL",
    "larsen": "LT", "larsen and toubro": "LT", "l&t": "LT",
    "zomato": "ZOMATO", "paytm": "PAYTM",
    "dmart": "DMART", "avenue supermarts": "DMART",
    "irctc": "IRCTC", "hal": "HAL", "hindustan aeronautics": "HAL", "bhel": "BHEL",
    # ------ US STOCKS ------
    "nvidia": "NVDA", "apple": "AAPL", "microsoft": "MSFT",
    "google": "GOOGL", "alphabet": "GOOGL", "amazon": "AMZN",
    "meta": "META", "facebook": "META", "tesla": "TSLA", "netflix": "NFLX",
    "amd": "AMD", "intel": "INTC", "disney": "DIS", "walmart": "WMT",
    "coca cola": "KO", "pepsi": "PEP", "pepsico": "PEP",
    "johnson and johnson": "JNJ", "jpmorgan": "JPM", "jp morgan": "JPM",
    "goldman sachs": "GS", "berkshire": "BRK-B", "berkshire hathaway": "BRK-B",
    "paypal": "PYPL", "salesforce": "CRM", "adobe": "ADBE",
    "uber": "UBER", "spotify": "SPOT", "snapchat": "SNAP", "snap": "SNAP",
    "airbnb": "ABNB", "palantir": "PLTR", "coinbase": "COIN",
    "snowflake": "SNOW", "crowdstrike": "CRWD", "shopify": "SHOP",
    "boeing": "BA", "ford": "F", "general motors": "GM",
    "visa": "V", "mastercard": "MA",
}


# ============================================================================
# INDEX MAPPINGS
# ============================================================================

INDICES = {
    "NIFTY": "NIFTY 50", "NIFTY50": "NIFTY 50", "NIFTY 50": "NIFTY 50",
    "SENSEX": "SENSEX", "BSE SENSEX": "SENSEX",
    "BANKNIFTY": "BANK NIFTY", "BANK NIFTY": "BANK NIFTY",
    "NIFTYIT": "NIFTY IT", "NIFTY IT": "NIFTY IT",
    "NIFTYFIN": "NIFTY FINANCIAL", "NIFTYPHARMA": "NIFTY PHARMA",
    "NIFTYAUTO": "NIFTY AUTO", "NIFTYMETAL": "NIFTY METAL",
    "NIFTYENERGY": "NIFTY ENERGY", "NIFTYFMCG": "NIFTY FMCG",
    "NIFTYNEXT50": "NIFTY NEXT 50", "NIFTYMIDCAP": "NIFTY MIDCAP",
    "MIDCAP100": "NIFTY MIDCAP 100",
}


# ============================================================================
# SYMBOL RESOLUTION
# ============================================================================

def resolve_symbol(name: str) -> Optional[str]:
    """
    Resolve a natural-language company name or ticker to a valid stock symbol.

    Tries in order:
    1. Exact match in COMPANY_NAME_MAP (case-insensitive)
    2. Direct uppercase match in NSE_KNOWN_SYMBOLS
    3. Fuzzy match via rapidfuzz (typo recovery)

    Returns:
        Resolved symbol string, or None if unresolvable.
    """
    if not name or not name.strip():
        return None

    name_clean = name.strip()
    name_lower = name_clean.lower()

    # Strategy 1: Exact company name match
    # Sort by length (longest first) to prefer "tata consultancy" over "tata"
    for key in sorted(COMPANY_NAME_MAP.keys(), key=len, reverse=True):
        if key in name_lower:
            return COMPANY_NAME_MAP[key]

    # Strategy 2: Direct symbol lookup (already uppercase)
    name_upper = name_clean.upper()
    if name_upper in NSE_KNOWN_SYMBOLS:
        return name_upper

    # Strategy 3: Fuzzy matching (typo recovery)
    try:
        from rapidfuzz import process as fuzz_process, fuzz

        # Build candidates: individual words + bigrams
        words_raw = name_lower.split()
        candidates = [w for w in words_raw if len(w) >= 4]
        for i in range(len(words_raw) - 1):
            bigram = f"{words_raw[i]} {words_raw[i + 1]}"
            if len(bigram) >= 4:
                candidates.append(bigram)
        for i in range(len(words_raw) - 2):
            trigram = f"{words_raw[i]} {words_raw[i + 1]} {words_raw[i + 2]}"
            candidates.append(trigram)

        best_score, best_symbol = 0, None
        name_keys = list(COMPANY_NAME_MAP.keys())

        for candidate in candidates:
            result = fuzz_process.extractOne(
                candidate, name_keys, scorer=fuzz.WRatio,
            )
            if result and result[1] > best_score:
                best_score = result[1]
                best_symbol = COMPANY_NAME_MAP[result[0]]

        if best_score >= 85 and best_symbol:
            logger.info(f"Fuzzy resolved: '{name}' → {best_symbol} (score={best_score})")
            return best_symbol

    except ImportError:
        import difflib
        name_keys = list(COMPANY_NAME_MAP.keys())
        matches = difflib.get_close_matches(name_lower, name_keys, n=1, cutoff=0.75)
        if matches:
            best_symbol = COMPANY_NAME_MAP[matches[0]]
            logger.info(f"Difflib resolved: '{name}' → {best_symbol}")
            return best_symbol
        
    return None


def resolve_index(name: str) -> Optional[str]:
    """Resolve an index name to its canonical form."""
    if not name:
        return None
    name_upper = name.strip().upper()
    return INDICES.get(name_upper)


---
# 7. Trading Knowledge Base (`chatbot/modules/trading_assistant.py`)

A **built-in knowledge base** with 18 trading topics — order types, strategies, technical analysis, fundamentals, how-to guides.

Used by the `search_knowledge_base` tool to answer educational queries without needing an LLM call.

In [ ]:
"""
Trading assistant module with comprehensive trading knowledge.
Provides Q&A for trading concepts, how-to guides, and educational content.
"""

from typing import Optional, List, Dict, Any
import json
import os
from pathlib import Path


# ============================================================================
# TRADING KNOWLEDGE BASE
# ============================================================================

TRADING_KNOWLEDGE = {
    # ORDER TYPES
    "market_order": {
        "title": "Market Order",
        "content": """A **Market Order** is an order to buy or sell a stock immediately at the best available current price.

**Key Points:**
• Executes instantly during market hours
• Price is not guaranteed - you get the current market price
• Best for: Highly liquid stocks when speed is priority
• Risk: Slippage in volatile markets

**Example:** If TCS is trading at ₹3,500, a market buy order will execute at approximately that price.""",
        "keywords": ["market order", "instant order", "immediate buy", "immediate sell"]
    },
    
    "limit_order": {
        "title": "Limit Order",
        "content": """A **Limit Order** lets you specify the maximum price for buying or minimum price for selling.

**Key Points:**
• Buy Limit: Order executes only at your price or lower
• Sell Limit: Order executes only at your price or higher
• May not execute if price doesn't reach your limit
• Gives you price control

**Example:** Set a buy limit at ₹3,400 for TCS - it only buys if price falls to ₹3,400 or below.""",
        "keywords": ["limit order", "price limit", "set price", "target price"]
    },
    
    "stop_loss": {
        "title": "Stop Loss Order",
        "content": """A **Stop Loss (SL)** order helps limit your losses by automatically selling when price falls to a certain level.

**Key Points:**
• Protects against large losses
• Triggers a market order when stop price is hit
• Essential for risk management
• Can also be used to protect profits (trailing SL)

**How to set:**
1. Go to your holdings/positions
2. Select the stock
3. Click 'Add Stop Loss'
4. Set your trigger price

**Example:** Bought stock at ₹100, set SL at ₹90 to limit loss to 10%.""",
        "keywords": ["stop loss", "stoploss", "sl", "cut loss", "limit loss", "protect"]
    },
    
    "bracket_order": {
        "title": "Bracket Order",
        "content": """A **Bracket Order** is an advanced order type that places 3 orders at once: entry, target, and stop-loss.

**Components:**
1. **Entry Order**: Your buy/sell order
2. **Target Order**: Profit booking level
3. **Stop Loss**: Loss limiting level

**Benefits:**
• Automated risk management
• Lock in profit targets
• Popular for intraday trading

**Note:** Only available for intraday positions on most platforms.""",
        "keywords": ["bracket order", "bo", "bracket", "three leg order"]
    },
    
    # TRADING CONCEPTS
    "intraday": {
        "title": "Intraday Trading",
        "content": """**Intraday Trading** means buying and selling stocks within the same trading day.

**Key Points:**
• All positions must be squared off before market close
• Requires less capital (leverage available)
• Higher risk, higher reward potential
• Profits/losses settled same day

**Tips for Beginners:**
• Start small
• Use stop losses always
• Focus on liquid stocks
• Avoid trading on news/events initially

**Timing:** NSE market hours are 9:15 AM - 3:30 PM IST.""",
        "keywords": ["intraday", "day trading", "same day", "square off"]
    },
    
    "delivery": {
        "title": "Delivery Trading",
        "content": """**Delivery Trading** means buying stocks to hold for more than one day.

**Key Points:**
• Stocks are delivered to your demat account
• No time limit - hold for days, months, or years
• Requires full payment (no leverage)
• Pay STT on buy and sell

**Benefits:**
• No daily monitoring needed
• Can benefit from long-term growth
• No forced square-off
• Eligible for dividends and bonuses""",
        "keywords": ["delivery", "cash", "cnc", "long term", "hold", "invest"]
    },
    
    "margin": {
        "title": "Margin Trading",
        "content": """**Margin Trading** allows you to trade with borrowed funds from your broker.

**Key Points:**
• Trade larger positions with less capital
• Amplifies both profits AND losses
• Interest charged on borrowed amount
• Margin call if position moves against you

**Types:**
• **Intraday Margin**: Up to 5x for intraday
• **Delivery Margin**: Usually 2x-4x

⚠️ **Risk Warning:** Margin trading is risky. You can lose more than your initial investment.""",
        "keywords": ["margin", "leverage", "borrowed", "mis", "margin trading"]
    },
    
    # TECHNICAL ANALYSIS
    "candlestick": {
        "title": "Candlestick Patterns",
        "content": """**Candlestick Charts** show price movement with visual candles.

**Anatomy of a Candle:**
• **Body**: Opening to closing price
• **Wicks/Shadows**: High and low prices
• **Green/White**: Closing higher than opening (bullish)
• **Red/Black**: Closing lower than opening (bearish)

**Common Patterns:**
• **Doji**: Indecision, opening = closing
• **Hammer**: Potential reversal at bottom
• **Shooting Star**: Potential reversal at top
• **Engulfing**: Strong reversal signal

**Tip:** Combine with volume and other indicators for confirmation.""",
        "keywords": ["candlestick", "candle", "pattern", "chart pattern", "doji", "hammer"]
    },
    
    "moving_average": {
        "title": "Moving Averages",
        "content": """**Moving Averages** smooth out price data to show trends.

**Types:**
• **SMA (Simple)**: Average of last N prices
• **EMA (Exponential)**: Gives more weight to recent prices

**Popular Periods:**
• 9 & 21 day: Short-term trends
• 50 day: Medium-term trend
• 200 day: Long-term trend

**Crossover Signals:**
• **Golden Cross**: 50 MA crosses above 200 MA (bullish)
• **Death Cross**: 50 MA crosses below 200 MA (bearish)

**Usage:** Price above MA = uptrend, below MA = downtrend.""",
        "keywords": ["moving average", "sma", "ema", "ma", "average", "trend"]
    },
    
    "rsi": {
        "title": "RSI (Relative Strength Index)",
        "content": """**RSI** is a momentum indicator measuring speed and change of price movements.

**Scale:** 0 to 100

**Key Levels:**
• **Above 70**: Overbought (potential sell signal)
• **Below 30**: Oversold (potential buy signal)
• **50**: Neutral level

**How to Use:**
• Look for divergences (price vs RSI)
• Combine with price action
• Don't trade solely on RSI
• Works best in ranging markets

**Period:** Standard is 14 periods.""",
        "keywords": ["rsi", "relative strength", "overbought", "oversold", "momentum"]
    },
    
    # FUNDAMENTAL TERMS
    "pe_ratio": {
        "title": "P/E Ratio (Price to Earnings)",
        "content": """**P/E Ratio** shows how much investors pay per rupee of company earnings.

**Formula:** P/E = Stock Price ÷ Earnings Per Share (EPS)

**Interpretation:**
• **High P/E (>30)**: Expensive, high growth expectations
• **Low P/E (<15)**: Cheap or low growth expectations
• **Compare within same industry**

**Types:**
• **Trailing P/E**: Based on past 12 months earnings
• **Forward P/E**: Based on expected future earnings

**Example:** Stock at ₹500 with EPS of ₹25 = P/E of 20.""",
        "keywords": ["pe ratio", "price to earnings", "p/e", "pe", "valuation", "earnings"]
    },
    
    "market_cap": {
        "title": "Market Capitalization",
        "content": """**Market Cap** is the total market value of a company's outstanding shares.

**Formula:** Market Cap = Share Price × Total Shares Outstanding

**Categories:**
• **Large Cap**: > ₹20,000 Cr (stable, lower risk)
• **Mid Cap**: ₹5,000 - ₹20,000 Cr (moderate risk/reward)
• **Small Cap**: < ₹5,000 Cr (higher risk/reward)

**Why It Matters:**
• Indicates company size
• Larger = more stable, less volatile
• Smaller = more growth potential but riskier""",
        "keywords": ["market cap", "market capitalization", "cap", "large cap", "mid cap", "small cap"]
    },
    
    # PLATFORM/HOW-TO
    "how_to_buy": {
        "title": "How to Buy Stocks",
        "content": """**Steps to Buy Stocks:**

1. **Search** for the stock by name or symbol
2. **Click Buy** button
3. **Select order type:**
   • Market Order (instant execution)
   • Limit Order (specify your price)
4. **Choose product type:**
   • CNC/Delivery (long-term)
   • MIS/Intraday (same day)
5. **Enter quantity** (number of shares)
6. **Review** your order details
7. **Swipe/Click** to confirm

**Tips:**
• Always check the price before confirming
• Use limit orders for better price control
• Set stop-loss for risk management""",
        "keywords": ["how to buy", "buy stock", "purchase", "place order", "buying"]
    },
    
    "how_to_sell": {
        "title": "How to Sell Stocks",
        "content": """**Steps to Sell Stocks:**

1. Go to **Portfolio/Holdings**
2. **Select** the stock you want to sell
3. **Click Sell**
4. **Choose order type:**
   • Market Order
   • Limit Order
5. **Enter quantity** to sell
6. **Review** the order
7. **Confirm** the sale

**For Intraday/Short Selling:**
• Go to the stock page
• Select "Sell" (without owning)
• Choose MIS product type
• Must buy back before market close""",
        "keywords": ["how to sell", "sell stock", "exit", "selling", "book profit"]
    },
    
    "t_plus_one": {
        "title": "T+1 Settlement",
        "content": """**T+1 Settlement** means trades are settled the next business day after the trade date.

India moved to T+1 settlement for all NSE/BSE stocks in 2023, making it one of the fastest settlement systems globally.

**What this means for traders:**
- Buy today, shares credited to demat account next trading day
- Sell today, money credited to bank account next trading day
- No more waiting 2 days like the old T+2 system
- Reduces counterparty risk significantly

**Intraday exception:** Intraday trades are squared off same day — no delivery involved.""",
        "keywords": ["t+1", "settlement", "t plus one", "trade settlement", "demat credit"]
    },

    "circuit_breaker": {
        "title": "Circuit Breakers and Price Bands",
        "content": """**Circuit Breakers** halt trading when markets move sharply to prevent panic.

**Market-wide circuit breakers (NSE/BSE):**
- 10% move → 45 minute halt (before 1 PM) / 15 minutes (between 1-2:30 PM) / no halt after 2:30 PM
- 15% move → 1 hour 45 minute halt / 45 minutes / no halt after 2:30 PM
- 20% move → Trading halted for rest of the day

**Stock-specific price bands:**
- 2%, 5%, 10%, or 20% upper/lower circuit limits
- Stock hitting upper circuit = only buyers, no sellers (very bullish signal)
- Stock hitting lower circuit = only sellers, no buyers (very bearish signal)
- F&O stocks: no price band (but index circuit breakers apply)""",
        "keywords": ["circuit breaker", "upper circuit", "lower circuit", "price band", "halt trading"]
    },

    "sebi_investor": {
        "title": "SEBI Investor Protection Rules",
        "content": """**SEBI (Securities and Exchange Board of India)** regulates Indian stock markets.

**Key investor protections:**
- Mandatory demat account for all share holdings
- Broker margins must be in segregated accounts
- SEBI SCORES: Online complaint portal for investor grievances
- Investor Protection Fund: Covers up to ₹25 lakh if broker defaults
- KYC mandatory for all trading accounts

**SEBI registration check:**
Always verify your broker/advisor is SEBI registered at sebi.gov.in/sebiweb/home/HomeAction.do

**Disclaimer:** FinSight is an informational tool. Always verify with SEBI-registered advisors.""",
        "keywords": ["SEBI", "investor protection", "broker default", "SCORES", "KYC", "regulation"]
    },

    "fii_dii": {
        "title": "FII and DII Activity",
        "content": """**FII (Foreign Institutional Investors)** and **DII (Domestic Institutional Investors)** are major market movers.

**FII:**
- Foreign funds investing in Indian markets
- When FIIs buy heavily = bullish signal, market tends to rise
- When FIIs sell (net sellers) = bearish signal, market tends to fall
- Tracked daily on NSE website

**DII:**
- Indian mutual funds, insurance companies, pension funds
- Often counter FII selling (buy when FIIs sell)
- DII buying during FII selloff = market support signal

**Where to check:**
NSE India publishes daily FII/DII data. Important for understanding whether institutional money is flowing in or out of Indian markets.""",
        "keywords": ["FII", "DII", "foreign investors", "institutional buying", "institutional selling", "net buy", "net sell"]
    },
}


class TradingAssistant:
    """
    Provides trading knowledge, how-to guides, and educational content.
    Uses the built-in knowledge base and optional RAG for enhanced responses.
    """
    
    def __init__(self):
        """Initialize trading assistant."""
        self.knowledge = TRADING_KNOWLEDGE
        self._build_keyword_index()
    
    def _build_keyword_index(self):
        """Build reverse index from keywords to topics."""
        self.keyword_index: Dict[str, List[str]] = {}
        
        for topic_id, topic in self.knowledge.items():
            for keyword in topic.get("keywords", []):
                keyword_lower = keyword.lower()
                if keyword_lower not in self.keyword_index:
                    self.keyword_index[keyword_lower] = []
                self.keyword_index[keyword_lower].append(topic_id)
    
    def search(self, query: str) -> Optional[Dict[str, Any]]:
        """
        Search for relevant trading knowledge.
        
        Args:
            query: User's query
            
        Returns:
            Matching topic or None
        """
        query_lower = query.lower()
        
        # Direct keyword match
        for keyword, topics in self.keyword_index.items():
            if keyword in query_lower:
                # Return the first matching topic
                topic_id = topics[0]
                return {
                    "id": topic_id,
                    **self.knowledge[topic_id]
                }
        
        # Fuzzy match on topic titles
        for topic_id, topic in self.knowledge.items():
            title_lower = topic["title"].lower()
            if any(word in query_lower for word in title_lower.split()):
                return {
                    "id": topic_id,
                    **topic
                }
        
        return None
    
    def get_topic(self, topic_id: str) -> Optional[Dict[str, Any]]:
        """Get a specific topic by ID."""
        if topic_id in self.knowledge:
            return {
                "id": topic_id,
                **self.knowledge[topic_id]
            }
        return None
    
    def format_response(self, topic: Dict[str, Any]) -> str:
        """Format a topic into a response."""
        return f"📚 **{topic['title']}**\n\n{topic['content']}"
    
    def get_all_topics(self) -> List[str]:
        """Get list of all available topics."""
        return list(self.knowledge.keys())
    
    def get_help_suggestions(self) -> str:
        """Get suggestions for what users can ask about."""
        return """💡 **I can help you with:**

**Order Types:**
• Market orders, limit orders, stop-loss
• Bracket orders, cover orders

**Trading Concepts:**
• Intraday vs Delivery trading
• Margin trading explained
• Short selling basics

**Technical Analysis:**
• Candlestick patterns
• Moving averages (SMA, EMA)
• RSI and other indicators

**Fundamentals:**
• P/E ratio, Market Cap
• EPS, Dividend Yield

**How-To Guides:**
• How to buy/sell stocks
• Setting stop-loss
• Reading charts

Just ask me anything! 🚀"""


# Singleton instance
_trading_assistant = None

def get_trading_assistant() -> TradingAssistant:
    """Get or create the trading assistant singleton."""
    global _trading_assistant
    if _trading_assistant is None:
        _trading_assistant = TradingAssistant()
    return _trading_assistant


---
# 8. Response Formatting (`chatbot/modules/market_formatter.py`)

Converts raw data into **beautiful, emoji-rich responses** with Indian number formatting (L/Cr).

**Features:** 🟢/🔴 color-coded changes, ₹/$ auto-detection, day's range, volume, history tables.

In [ ]:
"""
Response formatting for market data.
Converts raw data into beautiful, conversational responses.
"""

from typing import Optional, List
from common.models.schemas import StockPrice, IndexData, StockDetails, MarketMovers, StockHistory, Market


class MarketFormatter:
    """
    Format market data into user-friendly responses.
    Uses emojis and clear formatting for better readability.
    """
    
    @staticmethod
    def format_price(price: float, currency: str = "₹") -> str:
        """Format price with currency symbol and thousand separators."""
        if price >= 10000000:  # 1 Crore
            return f"{currency}{price/10000000:.2f} Cr"
        elif price >= 100000:  # 1 Lakh  
            return f"{currency}{price/100000:.2f} L"
        else:
            return f"{currency}{price:,.2f}"
    
    @staticmethod
    def format_change(change: float, change_percent: float) -> str:
        """Format price change with emoji indicator."""
        if change > 0:
            emoji = "🟢"
            sign = "+"
        elif change < 0:
            emoji = "🔴"
            sign = ""
        else:
            emoji = "⚪"
            sign = ""
        
        return f"{emoji} {sign}₹{abs(change):.2f} ({sign}{change_percent:.2f}%)"
    
    @staticmethod
    def format_stock_price(stock: StockPrice) -> str:
        """Format stock price into a conversational response."""
        change_emoji = "📈" if stock.change >= 0 else "📉"
        change_color = "🟢" if stock.change >= 0 else "🔴"
        sign = "+" if stock.change >= 0 else ""
        
        response = f"""**{stock.name or stock.symbol}** ({stock.symbol}) {change_emoji}

💰 **Current Price:** ₹{stock.price:,.2f}
{change_color} **Change:** {sign}₹{abs(stock.change):.2f} ({sign}{stock.change_percent:.2f}%)"""
        
        # Add day's range if available
        if stock.high and stock.low:
            response += f"\n📊 **Day's Range:** ₹{stock.low:,.2f} - ₹{stock.high:,.2f}"
        
        # Add volume if available
        if stock.volume:
            vol_formatted = MarketFormatter._format_volume(stock.volume)
            response += f"\n📦 **Volume:** {vol_formatted}"
        
        return response
    
    @staticmethod
    def format_index(index: IndexData) -> str:
        """Format index data into a conversational response."""
        change_emoji = "📈" if index.change >= 0 else "📉"
        change_color = "🟢" if index.change >= 0 else "🔴"
        sign = "+" if index.change >= 0 else ""
        
        return f"""**{index.name}** {change_emoji}

🎯 **Current Value:** {index.value:,.2f}
{change_color} **Change:** {sign}{abs(index.change):.2f} ({sign}{index.change_percent:.2f}%)"""
    
    @staticmethod
    def format_stock_details(details: StockDetails, price: Optional[StockPrice] = None) -> str:
        """Format detailed stock information."""
        response = f"""**{details.name}** ({details.symbol}) 📊

"""
        
        if details.sector:
            response += f"🏢 **Sector:** {details.sector}\n"
        
        if details.industry:
            response += f"🏭 **Industry:** {details.industry}\n"
        
        if details.market_cap:
            response += f"💎 **Market Cap:** {MarketFormatter.format_price(details.market_cap)}\n"
        
        if details.pe_ratio:
            response += f"📈 **P/E Ratio:** {details.pe_ratio:.2f}\n"
        
        if details.eps:
            response += f"💵 **EPS:** ₹{details.eps:.2f}\n"
        
        if details.dividend_yield:
            response += f"🎁 **Dividend Yield:** {details.dividend_yield*100:.2f}%\n"
        
        if details.week_52_high and details.week_52_low:
            response += f"📅 **52-Week Range:** ₹{details.week_52_low:,.2f} - ₹{details.week_52_high:,.2f}\n"
        
        if details.description:
            # Truncate long descriptions
            desc = details.description[:300] + "..." if len(details.description) > 300 else details.description
            response += f"\n📝 **About:** {desc}"
        
        return response
    
    @staticmethod
    def format_market_summary(indices: dict) -> str:
        """Format overall market summary."""
        response = "📊 **Market Summary**\n\n"
        
        for name, index in indices.items():
            change_emoji = "🟢" if index.change >= 0 else "🔴"
            sign = "+" if index.change >= 0 else ""
            response += f"**{name}:** {index.value:,.2f} {change_emoji} {sign}{index.change_percent:.2f}%\n"
        
        return response
    
    @staticmethod
    def format_quick_price(stock: StockPrice) -> str:
        """Format a quick one-line price response."""
        change_emoji = "🟢" if stock.change >= 0 else "🔴"
        sign = "+" if stock.change >= 0 else ""
        
        return f"{stock.symbol}: ₹{stock.price:,.2f} {change_emoji} {sign}{stock.change_percent:.2f}%"
    
    @staticmethod
    def _format_volume(volume: int) -> str:
        """Format volume with appropriate suffix."""
        if volume >= 10000000:
            return f"{volume/10000000:.2f} Cr"
        elif volume >= 100000:
            return f"{volume/100000:.2f} L"
        elif volume >= 1000:
            return f"{volume/1000:.1f}K"
        else:
            return str(volume)
    
    @staticmethod
    def format_error(symbol: str, error_type: str = "not_found") -> str:
        """Format error messages."""
        if error_type == "not_found":
            return f"❓ Sorry, I couldn't find data for **{symbol}**. Please check the symbol and try again."
        elif error_type == "market_closed":
            return f"🌙 Markets are currently closed. Showing last available price for **{symbol}**."
        elif error_type == "timeout":
            return f"⏱️ The request timed out. Please try again in a moment."
        elif error_type == "history_unavailable":
            return f"📊 Sorry, I couldn't fetch historical data for **{symbol}**. The stock might be newly listed or data is temporarily unavailable."
        else:
            return f"⚠️ There was an error fetching data for **{symbol}**. Please try again."
    
    @staticmethod
    def format_stock_history(history: StockHistory) -> str:
        """Format historical stock data into a readable table."""
        currency = "$" if history.market == Market.US else "₹"
        name_display = f"{history.name} ({history.symbol})" if history.name else history.symbol
        days_count = len(history.days)
        
        lines = [f"📊 **{name_display}** — Last {days_count} Trading Days\n"]
        
        for day in history.days:
            # Format change indicator
            if day.change_percent is not None:
                if day.change_percent >= 0:
                    change_str = f"🟢 +{day.change_percent:.2f}%"
                else:
                    change_str = f"🔴 {day.change_percent:.2f}%"
            else:
                change_str = "—"
            
            # Format the date nicely
            try:
                from datetime import datetime
                dt = datetime.strptime(day.date, '%Y-%m-%d')
                date_str = dt.strftime('%b %d')
            except Exception:
                date_str = day.date
            
            lines.append(
                f"📅 **{date_str}**: {currency}{day.close:,.2f}  {change_str}"
            )
        
        # Overall summary
        if history.overall_change_percent is not None:
            if history.overall_change_percent >= 0:
                overall = f"🟢 +{history.overall_change_percent:.2f}%"
            else:
                overall = f"🔴 {history.overall_change_percent:.2f}%"
            lines.append(f"\n**Overall Change**: {overall} over {days_count} days")
        
        # Add market label
        market_label = "US Market" if history.market == Market.US else "NSE"
        lines.append(f"\n_Source: {market_label} via yfinance_")
        
        return "\n".join(lines)


---
# 9. RAG Pipeline (`chatbot/rag_chain.py`)

**Retrieval-Augmented Generation** using FAISS vector store + HuggingFace embeddings.

Converts the trading knowledge base into embeddings for semantic search. Also fetches Wikipedia summaries for company info.

> **Note:** This cell requires `langchain-huggingface`, `faiss-cpu`, `sentence-transformers`. If not installed, it will skip gracefully.

In [ ]:
# This cell requires: pip install langchain-huggingface faiss-cpu sentence-transformers
try:
    import logging
    from langchain_community.vectorstores import FAISS
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_core.documents import Document
    
    logger = logging.getLogger(__name__)
    _rag_retriever = None
    
    def build_knowledge_retriever():
        global _rag_retriever
        if _rag_retriever is not None:
            return _rag_retriever
        try:
            from chatbot.modules.trading_assistant import TRADING_KNOWLEDGE
            docs = [Document(
                page_content=e.get("title","") + "\n" + e.get("content",""),
                metadata={"key": k, "title": e.get("title", k)}
            ) for k, e in TRADING_KNOWLEDGE.items()]
            splits = RecursiveCharacterTextSplitter(
                chunk_size=600, chunk_overlap=60
            ).split_documents(docs)
            vstore = FAISS.from_documents(
                splits, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
            )
            _rag_retriever = vstore.as_retriever(search_kwargs={"k": 2})
            logger.info(f"RAG retriever ready — {len(splits)} chunks from {len(docs)} topics")
            return _rag_retriever
        except Exception as e:
            logger.error(f"RAG build failed: {e}")
            return None
    
    def get_rag_retriever():
        return build_knowledge_retriever()
    
    def get_wikipedia_summary(company_name: str) -> str:
        try:
            import urllib.request, json, urllib.parse
            q = urllib.parse.quote(company_name)
            url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{q}"
            req = urllib.request.Request(url, headers={"User-Agent": "FinSight/1.0"})
            with urllib.request.urlopen(req, timeout=5) as r:
                return json.loads(r.read()).get("extract", "")[:500]
        except Exception:
            return ""
    
    print('[OK] rag_chain.py loaded successfully')
except ImportError as e:
    print(f'[SKIP] rag_chain.py - Missing package: {e}')
    print(f'       Install with: pip install langchain-huggingface faiss-cpu sentence-transformers')

---
# 10. 🧠 LangGraph Agent — The Brain (`chatbot/agent.py`)

This is the **heart of the project** — a LangGraph `StateGraph` ReAct agent that replaced ~2,375 lines of hardcoded routing.

**Architecture:**
- Custom `AgentState` with typed message list, intent tracking, session ID
- 9 `@tool` functions wrapping existing data services
- `AsyncSqliteSaver` for persistent conversation memory
- Analysis sub-graph with fan-out/fan-in for parallel data fetching
- Structured LLM output for follow-up suggestions
- Streaming support via `astream_events`
- Fallback node for graceful error handling

> **Note:** This cell requires `langgraph`, `langchain-groq`. If not installed, it will skip gracefully.

In [ ]:
# This cell requires: pip install langgraph langchain-groq langgraph-checkpoint-sqlite
try:
    """
    LangGraph Agent for the FinSight chatbot.
    
    Replaces the hardcoded intent classifier + if/elif router with a single
    LLM-driven agent that dynamically picks the right tool(s) for each query.
    
    Architecture:
        User Message → StateGraph agent node → LLM picks tool(s) → ToolNode
        executes → LLM synthesises final answer → ChatResponse
    
    Tools wrap existing data services (yfinance, Screener.in, news RSS, knowledge base).
    
    ## Changes Made
    1.  [ARCHITECTURE] Replaced create_react_agent with a full StateGraph using typed
        AgentState (TypedDict with Annotated message list, intent, tools_used, session_id).
    2.  [MEMORY] Replaced MemorySaver with AsyncSqliteSaver from
        langgraph.checkpoint.sqlite.aio using connection string "finsight.db".
        Added get_graph() async factory that lazily initialises and caches the graph.
    3.  [STREAMING] Added stream_message() async generator using astream_events v2.
        Yields token chunks on on_chat_model_stream and tool-start banners.
    4.  [TOOLS] Wrapped get_stock_price, get_index_data, get_market_summary,
        get_stock_details with response_format="content_and_artifact".
        Each returns tuple[str, dict] with symbol/value/timestamp artifact.
    5.  [ERROR_HANDLING] Added InjectedToolCallId to all tools. Replaced bare
        "❌ Could not find…" returns with raise ToolException(...). Configured
        ToolNode(handle_tool_errors=True). Removed manual 3-attempt retry loop.
    6.  [ERROR_HANDLING] Added conditional edge after agent node → tools / fallback / END.
        Fallback node catches Groq bad-format errors and returns a safe ChatResponse.
    7.  [PERFORMANCE] Converted get_stock_details to use asyncio.gather() for
        concurrent price + details + fundamentals fetching.
    8.  [PERFORMANCE] Extracted analyze_stock into a LangGraph sub-graph with
        fan-out (START → 3 fetch nodes in parallel) and fan-in (→ synthesise → END).
    9.  [ARCHITECTURE] Replaced regex _extract_suggestions() with structured LLM
        call using .with_structured_output() on AgentSuggestions Pydantic model.
    """
    
    import asyncio
    import logging
    import operator
    import uuid
    from typing import Optional, List, Annotated, TypedDict
    from datetime import datetime, timezone
    
    from langchain_groq import ChatGroq
    from langchain_core.tools import tool, ToolException, InjectedToolCallId
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from langgraph.graph import StateGraph, END, START
    from langgraph.graph.message import add_messages
    from langgraph.prebuilt import ToolNode
    # NOTE: Requires separate package `langgraph-checkpoint-sqlite` (pip install langgraph-checkpoint-sqlite)
    from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
    # NOTE: In langgraph >=0.2.x Send lives in langgraph.constants; some older
    # versions exported it from langgraph.types — adjust if your version differs.
    from langgraph.constants import Send
    from pydantic import BaseModel, Field
    
    from common.config.settings import settings
    from common.config.prompts import AGENT_SYSTEM_PROMPT
    from common.models.schemas import ChatResponse
    from chatbot.core.symbol_utils import resolve_symbol, resolve_index, COMPANY_NAME_MAP
    
    logger = logging.getLogger(__name__)
    
    
    # ============================================================================
    # STATE DEFINITIONS
    # ============================================================================
    
    class AgentState(TypedDict):
        """Typed state for the main agent graph."""
        messages: Annotated[list, add_messages]
        intent: str
        tools_used: list[str]
        session_id: str
    
    
    class AnalysisSubState(TypedDict):
        """State for the analyze_stock fan-out sub-graph."""
        symbol: str
        price_data: Annotated[list, operator.add]
        fundamentals_data: Annotated[list, operator.add]
        technicals_data: Annotated[list, operator.add]
        final_output: str
    
    
    class AgentSuggestions(BaseModel):
        """Structured output model for follow-up suggestions."""
        suggestions: list[str] = Field(default_factory=list, max_length=4)
    
    
    # ============================================================================
    # TOOL DEFINITIONS
    # Each tool wraps an existing data service and returns a formatted string.
    # The LLM reads the string and synthesises a user-facing response.
    # ============================================================================
    
    @tool(response_format="content_and_artifact")
    async def get_stock_price(
        symbol: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> tuple[str, dict]:
        """
        Get the current real-time price of a stock.
        Use this when the user asks for a stock's price, rate, CMP, LTP, or value.
        Accepts stock symbols (TCS, RELIANCE) or company names (Tata Motors, Infosys).
        """
        from common.data_services.market_data import get_market_data_service
        from chatbot.modules.market_formatter import MarketFormatter
    
        resolved = resolve_symbol(symbol) or symbol.upper()
        service = get_market_data_service()
        price = await service.get_stock_price(resolved)
    
        if price:
            content = MarketFormatter.format_stock_price(price)
            artifact = {
                "symbol": resolved,
                "value": price.price,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            return content, artifact
        raise ToolException(f"Could not find price data for '{symbol}'. Please check the stock symbol.")
    
    
    @tool(response_format="content_and_artifact")
    async def get_index_data(
        index_name: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> tuple[str, dict]:
        """
        Get the current value of a market index like NIFTY 50, SENSEX, or BANK NIFTY.
        Use this when the user asks about market indices.
        """
        from common.data_services.market_data import get_market_data_service
        from chatbot.modules.market_formatter import MarketFormatter
    
        resolved = resolve_index(index_name) or index_name.upper()
        service = get_market_data_service()
        data = await service.get_index_data(resolved)
    
        if data:
            content = MarketFormatter.format_index(data)
            artifact = {
                "symbol": resolved,
                "value": data.value,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            return content, artifact
        raise ToolException(f"Could not find data for index '{index_name}'.")
    
    
    @tool(response_format="content_and_artifact")
    async def get_market_summary(
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> tuple[str, dict]:
        """
        Get an overview of the entire Indian stock market — NIFTY 50, SENSEX, Bank NIFTY levels.
        Use this when the user asks 'how is the market today' or wants a market overview.
        """
        from common.data_services.market_data import get_market_data_service
        from chatbot.modules.market_formatter import MarketFormatter
    
        service = get_market_data_service()
        summary = await service.get_market_summary()
    
        if summary:
            content = MarketFormatter.format_market_summary(summary)
            artifact = {
                "symbol": "MARKET_OVERVIEW",
                "value": "summary",
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            return content, artifact
        raise ToolException("Unable to fetch market data at the moment.")
    
    
    @tool(response_format="content_and_artifact")
    async def get_stock_details(
        symbol: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> tuple[str, dict]:
        """
        Get detailed company information including sector, PE ratio, EPS, market cap,
        52-week range, and fundamental data from Screener.in.
        Use this when the user asks 'tell me about TCS', 'PE ratio of X', or wants company info.
        """
        from common.data_services.market_data import get_market_data_service
        from common.data_services.screener_in_service import get_screener_in_service
        from chatbot.modules.market_formatter import MarketFormatter
        from chatbot.rag_chain import get_wikipedia_summary
    
        resolved = resolve_symbol(symbol) or symbol.upper()
        service = get_market_data_service()
        screener = get_screener_in_service()
    
        # Upgrade 7: fetch concurrently with asyncio.gather
        details, price, fundamentals = await asyncio.gather(
            service.get_stock_details(resolved),
            service.get_stock_price(resolved),
            screener.get_fundamentals(resolved),
        )
    
        parts = []
    
        if price:
            parts.append(MarketFormatter.format_stock_price(price))
    
        if details:
            parts.append(MarketFormatter.format_stock_details(details, price))
            wiki = get_wikipedia_summary((details.name or resolved) + " company India")
            if wiki:
                parts.append(f"\n📖 **Wikipedia:** {wiki}")
    
        if fundamentals:
            parts.append(f"\n📊 **Screener.in Fundamentals:**\n{screener.format_for_llm(fundamentals)}")
    
        if parts:
            content = "\n\n".join(parts)
            artifact = {
                "symbol": resolved,
                "value": price.price if price else None,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            return content, artifact
        raise ToolException(f"Could not find details for '{symbol}'.")
    
    
    @tool
    async def get_stock_history(
        symbol: str,
        days: int,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> str:
        """
        Get historical stock price data (OHLCV) for the last N trading days.
        Use this when the user asks about past performance, price movement, trends over days/weeks.
        Use 5 for recent, 7 for 'last week', 30 for 'last month', 90 for 'last quarter'.
        """
        from common.data_services.market_data import get_market_data_service
        from chatbot.modules.market_formatter import MarketFormatter
    
        resolved = resolve_symbol(symbol) or symbol.upper()
        days = min(max(days, 1), 90)
    
        service = get_market_data_service()
        history = await service.get_stock_history(resolved, days)
    
        if history:
            return MarketFormatter.format_stock_history(history)
        raise ToolException(f"Could not fetch history for '{symbol}'.")
    
    
    @tool
    async def get_stock_news(
        query: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> str:
        """
        Get the latest financial news articles.
        Pass a stock symbol like 'TCS' or 'RELIANCE' for stock-specific news.
        Pass 'market' for general market news and headlines.
        """
        from common.data_services.news_service import get_news_service
    
        service = get_news_service()
    
        if query.lower() in ("market", "general", "latest", "all", "news"):
            articles = await service.get_market_news(limit=5)
        else:
            resolved = resolve_symbol(query) or query.upper()
            articles = await service.get_stock_news(resolved, limit=5)
    
        return service.format_news(articles)
    
    
    @tool
    def search_knowledge_base(
        query: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> str:
        """
        Search the trading knowledge base and Wikipedia for educational content.
        Use this when the user asks 'what is X', 'explain Y', or wants to learn
        about trading concepts like stop-loss, PE ratio, intraday, candlestick patterns, etc.
        """
        from chatbot.modules.trading_assistant import get_trading_assistant
        from chatbot.rag_chain import get_rag_retriever
        from common.data_services.wikipedia_service import get_wikipedia_service
    
        parts = []
    
        assistant = get_trading_assistant()
        topic = assistant.search(query)
        if topic:
            parts.append(assistant.format_response(topic))
    
        retriever = get_rag_retriever()
        if retriever:
            try:
                docs = retriever.invoke(query)
                if docs:
                    rag_text = "\n".join(d.page_content for d in docs[:2])
                    parts.append(f"📚 **Knowledge Base:**\n{rag_text}")
            except Exception:
                pass
    
        wiki_service = get_wikipedia_service()
        search_query = query.lower()
        for prefix in ("what is", "tell me about", "explain", "define"):
            search_query = search_query.replace(prefix, "").strip()
        search_query = search_query.rstrip("?")
    
        wiki_summary = wiki_service.search_concept(search_query) if search_query else None
        if wiki_summary:
            parts.append(wiki_service.format_for_llm(search_query, wiki_summary))
    
        if parts:
            return "\n\n".join(parts)
        return f"No specific knowledge found for '{query}'. Please answer from your financial expertise."
    
    
    @tool
    async def screen_stocks(
        screen_name: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> str:
        """
        Run a predefined stock screener to find stocks matching criteria.
        Available screens: 'undervalued', 'momentum', 'oversold', 'high_dividend', 'strong_fundamentals'.
        Use this when the user asks for 'undervalued stocks', 'momentum stocks', 'best stocks', etc.
        """
        from screener.screener import get_screener_service
        from screener.screener_formatter import ScreenerFormatter
    
        screener = get_screener_service()
    
        screen_map = {
            "undervalued": "undervalued", "cheap": "undervalued", "value": "undervalued",
            "momentum": "momentum", "trending": "momentum", "bullish": "momentum",
            "oversold": "oversold", "beaten down": "oversold", "dip": "oversold",
            "dividend": "high_dividend", "yield": "high_dividend", "income": "high_dividend",
            "strong fundamentals": "strong_fundamentals", "quality": "strong_fundamentals",
        }
    
        resolved_screen = screen_map.get(screen_name.lower(), screen_name.lower())
        result = await screener.get_prebuilt_screen(resolved_screen)
    
        if result:
            return ScreenerFormatter.format_screener_result(result)
    
        screens = screener.get_available_screens()
        lines = ["📊 **Available Stock Screens:**\n"]
        for s in screens:
            lines.append(f"• **{s['name']}** — {s['description']}")
        return "\n".join(lines)
    
    
    # ============================================================================
    # ANALYSIS SUB-GRAPH  (Upgrade 8)
    # Fan-out: START → fetch_price, fetch_fundamentals, fetch_technicals (parallel)
    # Fan-in:  all 3 → synthesise → END
    # ============================================================================
    
    async def _fetch_price_node(state: AnalysisSubState) -> dict:
        """Fetch current stock price."""
        try:
            from common.data_services.market_data import get_market_data_service
            service = get_market_data_service()
            price = await service.get_stock_price(state["symbol"])
            return {"price_data": [price]}
        except Exception as e:
            logger.error(f"Sub-graph fetch_price error: {e}")
            return {"price_data": [None]}
    
    
    async def _fetch_fundamentals_node(state: AnalysisSubState) -> dict:
        """Fetch fundamental data via screener service."""
        try:
            from common.data_services.market_data import get_market_data_service
            from screener.screener import get_screener_service
            service = get_market_data_service()
            yf_symbol, _ = service._detect_market(state["symbol"])
            screener = get_screener_service()
            fundamental = await screener._get_fundamentals(yf_symbol, state["symbol"])
            return {"fundamentals_data": [fundamental]}
        except Exception as e:
            logger.error(f"Sub-graph fetch_fundamentals error: {e}")
            return {"fundamentals_data": [None]}
    
    
    def _fetch_technicals_node(state: AnalysisSubState) -> dict:
        """Fetch technical indicators (sync — runs in thread pool)."""
        try:
            from common.data_services.market_data import get_market_data_service
            from screener.technical_indicators import get_indicator_service
            service = get_market_data_service()
            yf_symbol, _ = service._detect_market(state["symbol"])
            indicators = get_indicator_service()
            tech_data = indicators.get_all_indicators(yf_symbol, period="6mo")
            return {"technicals_data": [tech_data]}
        except Exception as e:
            logger.error(f"Sub-graph fetch_technicals error: {e}")
            return {"technicals_data": [{}]}
    
    
    async def _synthesise_node(state: AnalysisSubState) -> dict:
        """Combine fetched data into a formatted analysis string."""
        from screener.screener_formatter import ScreenerFormatter
        from common.data_services.market_data import get_market_data_service
        from common.models.schemas import (
            StockAnalysis, TechnicalIndicators, FundamentalData, Market,
        )
    
        symbol = state["symbol"]
        price = state["price_data"][0] if state.get("price_data") else None
        tech_data = state["technicals_data"][0] if state.get("technicals_data") else {}
        fundamental = state["fundamentals_data"][0] if state.get("fundamentals_data") else None
    
        if not tech_data and not price:
            return {"final_output": ""}
    
        service = get_market_data_service()
        _, detected_market = service._detect_market(symbol)
        market_enum = Market.US if detected_market == "US" else Market.NSE
    
        technical = TechnicalIndicators(
            rsi=tech_data.get("rsi", {}).get("value"),
            sma_20=tech_data.get("sma_20", {}).get("value"),
            sma_50=tech_data.get("sma_50", {}).get("value"),
            sma_200=tech_data.get("sma_200", {}).get("value"),
            ema_12=tech_data.get("ema_12", {}).get("value"),
            ema_26=tech_data.get("ema_26", {}).get("value"),
            macd=tech_data.get("macd", {}).get("macd"),
            macd_signal=tech_data.get("macd", {}).get("signal_line"),
            macd_histogram=tech_data.get("macd", {}).get("histogram"),
            bollinger_upper=tech_data.get("bollinger", {}).get("upper"),
            bollinger_lower=tech_data.get("bollinger", {}).get("lower"),
            bollinger_position=tech_data.get("bollinger", {}).get("band_position"),
            volume_ratio=tech_data.get("volume_analysis", {}).get("volume_ratio"),
            supertrend_signal=tech_data.get("supertrend", {}).get("signal"),
            adx=tech_data.get("adx", {}).get("value"),
            vwap_position=tech_data.get("vwap", {}).get("signal"),
            stochastic_k=tech_data.get("stochastic", {}).get("k"),
        )
    
        current_price = tech_data.get("current_price", 0)
        if current_price == 0 and price:
            current_price = price.price
        change_pct = price.change_percent if price else 0
        stock_name = price.name if price else symbol
    
        analysis = StockAnalysis(
            symbol=symbol.upper(),
            name=stock_name,
            price=current_price,
            change_percent=change_pct,
            technical=technical,
            fundamental=fundamental or FundamentalData(),
            signal=tech_data.get("composite_signal", "NEUTRAL"),
            score=tech_data.get("composite_score", 50.0),
            market=market_enum,
        )
        return {"final_output": ScreenerFormatter.format_analysis(analysis)}
    
    
    def _dispatch_analysis(state: AnalysisSubState):
        """Fan-out: send state to all 3 fetch nodes in parallel."""
        return [
            Send("fetch_price", state),
            Send("fetch_fundamentals", state),
            Send("fetch_technicals", state),
        ]
    
    
    _analysis_subgraph = None
    
    
    def _get_analysis_subgraph():
        """Build and cache the analysis sub-graph."""
        global _analysis_subgraph
        if _analysis_subgraph is None:
            builder = StateGraph(AnalysisSubState)
            builder.add_node("fetch_price", _fetch_price_node)
            builder.add_node("fetch_fundamentals", _fetch_fundamentals_node)
            builder.add_node("fetch_technicals", _fetch_technicals_node)
            builder.add_node("synthesise", _synthesise_node)
    
            builder.add_conditional_edges(START, _dispatch_analysis)
            builder.add_edge("fetch_price", "synthesise")
            builder.add_edge("fetch_fundamentals", "synthesise")
            builder.add_edge("fetch_technicals", "synthesise")
            builder.add_edge("synthesise", END)
    
            _analysis_subgraph = builder.compile()
        return _analysis_subgraph
    
    
    @tool
    async def analyze_stock(
        symbol: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> str:
        """
        Run a full technical + fundamental analysis on a single stock.
        Returns composite score, signal (BUY/SELL/HOLD), RSI, MACD, PE ratio, etc.
        Use this when the user asks to 'analyze X', 'full analysis of X', 'should I look at X'.
        """
        resolved = resolve_symbol(symbol) or symbol.upper()
        subgraph = _get_analysis_subgraph()
        result = await subgraph.ainvoke({
            "symbol": resolved,
            "price_data": [],
            "fundamentals_data": [],
            "technicals_data": [],
            "final_output": "",
        })
    
        if result.get("final_output"):
            return result["final_output"]
        raise ToolException(f"Could not analyze '{symbol}'. Please try another stock.")
    
    
    # ============================================================================
    # TOOL → INTENT MAPPING  (unchanged)
    # ============================================================================
    
    TOOL_INTENT_MAP = {
        "get_stock_price": "MARKET_PRICE",
        "get_index_data": "MARKET_TREND",
        "get_market_summary": "MARKET_TREND",
        "get_stock_details": "STOCK_INFO",
        "get_stock_history": "STOCK_HISTORY",
        "get_stock_news": "NEWS_REQUEST",
        "search_knowledge_base": "EDUCATION",
        "screen_stocks": "STOCK_SCREEN",
        "analyze_stock": "STOCK_SCREEN",
    }
    
    
    # ============================================================================
    # ALL TOOLS LIST
    # ============================================================================
    
    ALL_TOOLS = [
        get_stock_price,
        get_index_data,
        get_market_summary,
        get_stock_details,
        get_stock_history,
        get_stock_news,
        search_knowledge_base,
        screen_stocks,
        analyze_stock,
    ]
    
    
    # ============================================================================
    # MAIN GRAPH CONSTRUCTION
    # ============================================================================
    
    def _build_main_graph(llm: ChatGroq, checkpointer):
        """Build and compile the main agent StateGraph."""
    
        llm_with_tools = llm.bind_tools(ALL_TOOLS)
    
        # --- nodes -----------------------------------------------------------
    
        async def agent_node(state: AgentState) -> dict:
            """Invoke the LLM with bound tools."""
            try:
                system_msg = SystemMessage(content=AGENT_SYSTEM_PROMPT)
                response = await llm_with_tools.ainvoke(
                    [system_msg] + list(state["messages"])
                )
                return {"messages": [response]}
            except Exception as e:
                error_str = str(e)
                if any(kw in error_str for kw in [
                    "Failed to call a function",
                    "failed_generation",
                ]):
                    marker = AIMessage(
                        content=f"__GROQ_FORMAT_ERROR__:{error_str[:200]}"
                    )
                    return {"messages": [marker]}
                raise
    
        async def fallback_node(state: AgentState) -> dict:
            """Return a safe message when Groq produces malformed tool calls."""
            safe = AIMessage(
                content=(
                    "I apologize, but I encountered a temporary processing issue. "
                    "Please try rephrasing your question or try again in a moment."
                )
            )
            return {"messages": [safe]}
    
        # --- routing ---------------------------------------------------------
    
        def route_after_agent(state: AgentState) -> str:
            last = state["messages"][-1]
            if isinstance(last, AIMessage):
                if (isinstance(last.content, str)
                        and last.content.startswith("__GROQ_FORMAT_ERROR__")):
                    return "fallback"
                if getattr(last, "tool_calls", None):
                    return "tools"
            return END
    
        # --- assemble --------------------------------------------------------
    
        tools_node = ToolNode(ALL_TOOLS, handle_tool_errors=True)
    
        builder = StateGraph(AgentState)
        builder.add_node("agent", agent_node)
        builder.add_node("tools", tools_node)
        builder.add_node("fallback", fallback_node)
    
        builder.add_edge(START, "agent")
        builder.add_conditional_edges(
            "agent",
            route_after_agent,
            {"tools": "tools", "fallback": "fallback", END: END},
        )
        builder.add_edge("tools", "agent")   # loop back for multi-step
        builder.add_edge("fallback", END)
    
        return builder.compile(checkpointer=checkpointer)
    
    
    # ============================================================================
    # FinSightAgent CLASS
    # ============================================================================
    
    class FinSightAgent:
        """
        LangGraph-powered FinSight agent.
        Uses a custom StateGraph with an agent node, tool node, and fallback node.
        Must call ``await initialize()`` before first use to set up the async
        SQLite checkpointer.
        """
    
        def __init__(self):
            self.llm = ChatGroq(
                groq_api_key=settings.groq_api_key,
                model_name=settings.llm_model,
                temperature=settings.llm_temperature,
                max_tokens=settings.llm_max_tokens,
            )
            self.graph = None
            self._saver_cm = None  # async context-manager handle
    
        # --- lifecycle -------------------------------------------------------
    
        async def initialize(self):
            """Compile the graph inside an async context manager (required for
            AsyncSqliteSaver). Call once at application startup."""
            if self.graph is not None:
                return
            self._saver_cm = AsyncSqliteSaver.from_conn_string("finsight.db")
            saver = await self._saver_cm.__aenter__()
            self.graph = _build_main_graph(self.llm, saver)
            logger.info(
                f"✅ FinSight LangGraph agent ready — {len(ALL_TOOLS)} tools loaded"
            )
    
        async def shutdown(self):
            """Close the SQLite checkpointer connection."""
            if self._saver_cm:
                await self._saver_cm.__aexit__(None, None, None)
                self._saver_cm = None
    
        # --- main entry point ------------------------------------------------
    
        async def process_message(
            self,
            message: str,
            session_id: Optional[str] = None,
        ) -> ChatResponse:
            """Process a user message through the LangGraph agent."""
            session_id = session_id or str(uuid.uuid4())
    
            try:
                config = {"configurable": {"thread_id": session_id}}
                result = await self.graph.ainvoke(
                    {"messages": [HumanMessage(content=message)]},
                    config=config,
                )
    
                # Walk messages for tools used and final reply
                messages = result.get("messages", [])
                reply = ""
                tools_used: list[str] = []
    
                for msg in messages:
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        for tc in msg.tool_calls:
                            tools_used.append(tc.get("name", ""))
                    if isinstance(msg, AIMessage) and not getattr(msg, "tool_calls", None):
                        reply = msg.content
    
                if not reply:
                    reply = (
                        messages[-1].content if messages
                        else "I couldn't process your request."
                    )
    
                intent = self._derive_intent(tools_used, message)
                suggestions = await self._get_suggestions(reply)
    
                return ChatResponse(
                    reply=reply,
                    intent=intent,
                    suggestions=suggestions,
                    session_id=session_id,
                )
    
            except Exception as e:
                logger.error(f"Agent error: {e}", exc_info=True)
                return ChatResponse(
                    reply=(
                        "I apologize, but I encountered an error. "
                        f"Please try again. (Error: {str(e)[:100]})"
                    ),
                    intent="ERROR",
                    session_id=session_id,
                )
    
        # --- streaming entry point (Upgrade 3) -------------------------------
    
        async def stream_message(
            self,
            message: str,
            session_id: Optional[str] = None,
        ):
            """Async generator that yields token chunks and tool-start banners.
    
            Usage::
    
                async for chunk in agent.stream_message("price of TCS"):
                    print(chunk, end="", flush=True)
            """
            session_id = session_id or str(uuid.uuid4())
            config = {"configurable": {"thread_id": session_id}}
    
            async for event in self.graph.astream_events(
                {"messages": [HumanMessage(content=message)]},
                config=config,
                version="v2",
            ):
                kind = event["event"]
                if kind == "on_chat_model_stream":
                    chunk_content = event["data"]["chunk"].content
                    if chunk_content:
                        yield chunk_content
                elif kind == "on_tool_start":
                    tool_name = event.get("name", "tool")
                    yield f"\n⚙️ Using {tool_name}...\n"
                elif kind == "on_tool_end":
                    tool_output = event["data"].get("output", "")
                    # ToolMessage objects have a .content attribute
                    if hasattr(tool_output, "content"):
                        tool_output = tool_output.content
                    if isinstance(tool_output, str) and tool_output.strip():
                        yield f"\n{tool_output}\n\n"
    
        # --- helpers (unchanged) ---------------------------------------------
    
        def _derive_intent(self, tools_used: List[str], message: str) -> str:
            """Map tool usage back to an intent string for the UI badge."""
            if not tools_used:
                msg_lower = message.lower().strip()
                greetings = [
                    "hi", "hello", "hey", "thanks", "bye", "good morning",
                    "good afternoon", "good evening", "how are you",
                ]
                if any(g in msg_lower for g in greetings):
                    return "GREETING"
                return "GENERAL"
            primary_tool = tools_used[0]
            return TOOL_INTENT_MAP.get(primary_tool, "GENERAL")
    
        # --- structured suggestions (Upgrade 9) ------------------------------
    
        async def _get_suggestions(self, reply: str) -> List[str]:
            """Extract follow-up suggestions via structured LLM output."""
            try:
                structured_llm = self.llm.with_structured_output(AgentSuggestions)
                result = await structured_llm.ainvoke(
                    "Based on this financial assistant response, suggest 2-3 "
                    "relevant follow-up questions the user might want to ask. "
                    "Keep them concise and actionable.\n\n"
                    f"Response:\n{reply[:500]}"
                )
                return result.suggestions[:4]
            except Exception:
                logger.debug("Structured suggestions extraction failed", exc_info=True)
                return []
    
    
    # ============================================================================
    # SINGLETONS
    # ============================================================================
    
    _agent: Optional[FinSightAgent] = None
    
    
    def get_agent() -> FinSightAgent:
        """Get or create the FinSight agent singleton.
        NOTE: You must call ``await get_agent().initialize()`` once at startup
        before invoking process_message / stream_message."""
        global _agent
        if _agent is None:
            _agent = FinSightAgent()
        return _agent
    
    
    _graph_cache = None
    
    
    async def get_graph():
        """Async factory that lazily initialises and caches the compiled graph."""
        global _graph_cache
        if _graph_cache is None:
            agent = get_agent()
            if agent.graph is None:
                await agent.initialize()
            _graph_cache = agent.graph
        return _graph_cache
    
    print('[OK] agent.py loaded successfully')
except ImportError as e:
    print(f'[SKIP] agent.py - Missing package: {e}')
    print(f'       Install with: pip install langgraph langchain-groq langgraph-checkpoint-sqlite')

---
# 11. FastAPI Server (`main.py`)

The **web server** that exposes the chatbot as REST, WebSocket, and SSE streaming endpoints.

**Endpoints:**
| Method | Path | Purpose |
|--------|------|---------|
| GET | `/health` | Health check |
| POST | `/chat` | Main chat (LangGraph agent) |
| POST | `/v2/chat` | Chat with full request model |
| POST | `/stream` | SSE streaming chat |
| WS | `/ws/chat` | WebSocket real-time chat |
| GET | `/market/{symbol}` | Direct stock price |
| GET | `/index/{name}` | Index data |
| GET | `/news` | Financial news |
| GET | `/analyze/{symbol}` | Full stock analysis |
| GET | `/screener/{screen}` | Run pre-built screen |

**Features:** CORS, rate limiting (slowapi), Sentry error monitoring, lifespan management.

> **Note:** This cell requires `fastapi`, `slowapi`, `sentry-sdk`. If not installed, it will skip gracefully.

In [ ]:
# This cell requires: pip install fastapi slowapi sentry-sdk
try:
    """
    FastAPI application for the FinSight Chatbot.
    Powered by a LangGraph ReAct agent with tool-calling.
    Includes REST and WebSocket endpoints, health checks, and CORS support.
    """
    
    import json
    import logging
    import uuid
    from contextlib import asynccontextmanager
    from typing import Optional
    
    from fastapi import FastAPI, HTTPException, WebSocket, WebSocketDisconnect, Request
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import StreamingResponse
    from pydantic import BaseModel
    from slowapi import Limiter, _rate_limit_exceeded_handler
    from slowapi.util import get_remote_address
    from slowapi.errors import RateLimitExceeded
    import sentry_sdk
    from sentry_sdk.integrations.fastapi import FastApiIntegration
    
    from common.config.settings import settings
    from chatbot.agent import get_agent, get_graph
    from common.models.schemas import ChatRequest, ChatResponse, HealthResponse
    
    
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
    logger = logging.getLogger(__name__)
    
    
    # ============================================================================
    # LIFESPAN MANAGEMENT
    # ============================================================================
    
    @asynccontextmanager
    async def lifespan(app: FastAPI):
        """Application lifespan manager for startup/shutdown."""
        # Startup
        if settings.sentry_dsn:
            sentry_sdk.init(
                dsn=settings.sentry_dsn,
                integrations=[FastApiIntegration()],
                traces_sample_rate=0.1,
            )
            logger.info("🛡️ Sentry error monitoring enabled")
        logger.info("🚀 Starting FinSight Chatbot API...")
        logger.info(f"📊 Default market: {settings.default_market}")
        logger.info(f"🤖 LLM Model: {settings.llm_model}")
        
        # Initialize LangGraph agent with async SQLite checkpointer
        agent = get_agent()
        await agent.initialize()
        logger.info("✅ FinSight LangGraph agent initialized")
        
        yield
        
        # Shutdown — close the SQLite checkpointer connection
        await agent.shutdown()
        logger.info("👋 Shutting down FinSight Chatbot API...")
    
    
    # ============================================================================
    # APP INITIALIZATION
    # ============================================================================
    
    app = FastAPI(
        title="FinSight Chatbot API",
        description="AI-powered financial assistant using LangGraph agent architecture",
        version="3.0.0",
        lifespan=lifespan
    )
    
    # Rate Limiting
    limiter = Limiter(key_func=get_remote_address)
    app.state.limiter = limiter
    app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
    
    # CORS Configuration
    app.add_middleware(
        CORSMiddleware,
        allow_origins=settings.cors_origins,
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )
    
    
    # ============================================================================
    # REQUEST MODELS
    # ============================================================================
    
    class MessageInput(BaseModel):
        """Simple message input for backward compatibility."""
        message: str
        session_id: Optional[str] = None
    
    
    # ============================================================================
    # ENDPOINTS
    # ============================================================================
    
    @app.get("/", response_model=HealthResponse)
    async def root():
        """Root endpoint with API info."""
        return HealthResponse(
            status="healthy",
            version="3.0.0"
        )
    
    
    @app.get("/health", response_model=HealthResponse)
    async def health_check():
        """Health check endpoint."""
        return HealthResponse(
            status="healthy",
            version="3.0.0"
        )
    
    
    @app.post("/chat", response_model=ChatResponse)
    @limiter.limit("20/minute")
    async def chat_endpoint(request: Request, input: MessageInput):
        """
        Main chat endpoint — powered by LangGraph ReAct agent.
        
        The agent dynamically selects the right tool(s) to answer the query:
        stock prices, news, analysis, education, screening, and more.
        """
        try:
            if not input.message or not input.message.strip():
                raise HTTPException(status_code=400, detail="Message cannot be empty")
            
            agent = get_agent()
            response = await agent.process_message(
                message=input.message.strip(),
                session_id=input.session_id
            )
            
            return response
            
        except Exception as e:
            logger.error(f"Chat error: {e}")
            raise HTTPException(status_code=500, detail=str(e))
    
    
    @app.post("/v2/chat", response_model=ChatResponse)
    @limiter.limit("20/minute")
    async def chat_v2_endpoint(request: Request, request_body: ChatRequest):
        """V2 chat endpoint with full request model."""
        try:
            agent = get_agent()
            response = await agent.process_message(
                message=request_body.message.strip(),
                session_id=request_body.session_id
            )
            return response
            
        except Exception as e:
            logger.error(f"Chat error: {e}")
            raise HTTPException(status_code=500, detail=str(e))
    
    
    # ============================================================================
    # STREAMING ENDPOINT
    # ============================================================================
    
    @app.post("/stream")
    @limiter.limit("20/minute")
    async def stream_chat(request: Request, input: MessageInput):
        """
        Streaming chat endpoint — returns Server-Sent Events (SSE).
    
        Events:
          - {type: "token", content: "..."} — LLM output tokens
          - {type: "status", content: "..."} — tool usage notifications
          - {type: "done", intent: "...", suggestions: [...], session_id: "..."}
        """
        if not input.message or not input.message.strip():
            raise HTTPException(status_code=400, detail="Message cannot be empty")
    
        agent = get_agent()
        session_id = input.session_id or str(uuid.uuid4())
    
        async def event_generator():
            full_reply_parts = []
            tools_used = []
    
            async for chunk in agent.stream_message(
                message=input.message.strip(),
                session_id=session_id,
            ):
                full_reply_parts.append(chunk)
                stripped = chunk.strip()
    
                if stripped.startswith("\u2699\ufe0f Using ") and stripped.endswith("..."):
                    tool_name = stripped.replace("\u2699\ufe0f Using ", "").rstrip(".")
                    tools_used.append(tool_name)
                    yield f"data: {json.dumps({'type': 'status', 'content': stripped})}\n\n"
                else:
                    yield f"data: {json.dumps({'type': 'token', 'content': chunk})}\n\n"
    
            full_reply = "".join(full_reply_parts)
            intent = agent._derive_intent(tools_used, input.message.strip())
            try:
                suggestions = await agent._get_suggestions(full_reply[:500])
            except Exception:
                suggestions = []
    
            yield f"data: {json.dumps({'type': 'done', 'intent': intent, 'suggestions': suggestions, 'session_id': session_id})}\n\n"
    
        return StreamingResponse(
            event_generator(),
            media_type="text/event-stream",
            headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
        )
    
    
    # ============================================================================
    # WEBSOCKET ENDPOINT
    # ============================================================================
    
    @app.websocket("/ws/chat")
    async def websocket_chat(websocket: WebSocket):
        """WebSocket endpoint for real-time chat via LangGraph agent."""
        await websocket.accept()
        session_id = None
        agent = get_agent()
        
        try:
            while True:
                data = await websocket.receive_json()
                message = data.get("message", "").strip()
                session_id = data.get("session_id", session_id)
                
                if not message:
                    await websocket.send_json({"error": "Empty message"})
                    continue
                
                # Process message via LangGraph agent
                response = await agent.process_message(
                    message=message,
                    session_id=session_id
                )
                
                # Send response
                await websocket.send_json({
                    "reply": response.reply,
                    "intent": response.intent,
                    "entities": response.entities,
                    "suggestions": response.suggestions,
                    "session_id": response.session_id
                })
                
                # Update session_id for continuity
                session_id = response.session_id
                
        except WebSocketDisconnect:
            logger.info(f"WebSocket disconnected: session {session_id}")
        except Exception as e:
            logger.error(f"WebSocket error: {e}")
            await websocket.close()
    
    
    # ============================================================================
    # DIRECT DATA ENDPOINTS
    # ============================================================================
    
    @app.get("/market/{symbol}")
    async def get_stock_price(symbol: str):
        """Get current price for a stock symbol."""
        from common.data_services.market_data import get_market_data_service
        from chatbot.modules.market_formatter import MarketFormatter
        
        service = get_market_data_service()
        price = await service.get_stock_price(symbol.upper())
        
        if price:
            return {
                "symbol": price.symbol,
                "price": price.price,
                "change": price.change,
                "change_percent": price.change_percent,
                "formatted": MarketFormatter.format_quick_price(price)
            }
        
        raise HTTPException(status_code=404, detail=f"Stock {symbol} not found")
    
    
    @app.get("/index/{index_name}")
    async def get_index_data(index_name: str):
        """Get current data for a market index."""
        from common.data_services.market_data import get_market_data_service
        
        service = get_market_data_service()
        data = await service.get_index_data(index_name.upper())
        
        if data:
            return {
                "name": data.name,
                "value": data.value,
                "change": data.change,
                "change_percent": data.change_percent
            }
        
        raise HTTPException(status_code=404, detail=f"Index {index_name} not found")
    
    
    @app.get("/news")
    async def get_news(symbol: Optional[str] = None, limit: int = 5):
        """Get financial news (optionally filtered by stock symbol)."""
        from common.data_services.news_service import get_news_service
        
        service = get_news_service()
        
        if symbol:
            articles = await service.get_stock_news(symbol.upper(), limit)
        else:
            articles = await service.get_market_news(limit)
        
        return {
            "articles": [
                {
                    "title": a.title,
                    "summary": a.summary,
                    "source": a.source,
                    "url": a.url,
                    "published_at": a.published_at.isoformat()
                }
                for a in articles
            ]
        }
    
    
    # ============================================================================
    # SCREENER ENDPOINTS
    # ============================================================================
    
    @app.get("/screener/screens")
    async def list_screens():
        """List all available pre-built screener screens."""
        from screener.screener import get_screener_service
        screener = get_screener_service()
        return {"screens": screener.get_available_screens()}
    
    
    @app.get("/analyze/{symbol}")
    async def analyze_stock(symbol: str):
        """Get full technical + fundamental analysis for a stock."""
        from screener.screener import get_screener_service
        screener = get_screener_service()
        analysis = await screener.analyze_stock(symbol)
        if not analysis:
            return {"error": f"Could not analyze {symbol}"}
        return analysis.model_dump()
    
    
    @app.get("/screener/{screen_name}")
    async def run_screen(screen_name: str):
        """
        Run a pre-built stock screen.
        Options: undervalued, momentum, oversold, high_dividend, strong_fundamentals
        """
        from screener.screener import get_screener_service
        screener = get_screener_service()
        result = await screener.get_prebuilt_screen(screen_name)
        if not result:
            return {"error": f"Unknown screen: {screen_name}", "available": list(screener.get_available_screens())}
        return result.model_dump()
    
    
    class CustomScreenInput(BaseModel):
        filters: dict
        stock_list: Optional[list] = None
    
    
    @app.post("/screener/custom")
    async def custom_screen(input: CustomScreenInput):
        """Run a custom stock screen with user-defined filters."""
        from screener.screener import get_screener_service
        screener = get_screener_service()
        result = await screener.screen_stocks(
            filters=input.filters,
            stock_list=input.stock_list,
            screen_name="Custom Screen",
        )
        return result.model_dump()
    
    
    # ============================================================================
    # MAIN ENTRY POINT
    # ============================================================================
    
    if __name__ == "__main__":
        import uvicorn
        import os
        port = int(os.environ.get("PORT", settings.server_port))
        uvicorn.run(
            "main:app",
            host=settings.server_host,
            port=port,
            reload=False
        )
    
    print('[OK] main.py loaded successfully')
except ImportError as e:
    print(f'[SKIP] main.py - Missing package: {e}')
    print(f'       Install with: pip install fastapi slowapi sentry-sdk')

---
# 12. Streamlit Frontend (`streamlit_app.py`)

The **chat UI** built with Streamlit. Features intent badges, suggestion chips, and session continuity.

Run with: `streamlit run streamlit_app.py`

> **Note:** This cell requires `streamlit`. If not installed, it will skip gracefully.

In [ ]:
# This cell requires: pip install streamlit
try:
    """
    FinSight Chatbot — Streamlit Frontend
    Premium dark-themed chat interface for the FinSight financial assistant.
    Connects to the FastAPI backend at http://localhost:8000.
    """
    
    import streamlit as st
    import requests
    import json
    import time
    from datetime import datetime
    
    import os
    
    # ============================================================================
    # CONFIGURATION
    # ============================================================================
    
    API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8000")
    APP_TITLE = "FinSight"
    APP_SUBTITLE = "AI-Powered Financial Assistant"
    
    # Intent → emoji + color mapping
    INTENT_BADGES = {
        "MARKET_PRICE": ("💰", "#10b981"),
        "MARKET_TREND": ("📊", "#6366f1"),
        "STOCK_INFO": ("🏢", "#8b5cf6"),
        "STOCK_HISTORY": ("📈", "#f59e0b"),
        "STOCK_SCREEN": ("🔍", "#ec4899"),
        "TRADING_HOW_TO": ("📚", "#14b8a6"),
        "PORTFOLIO_QUERY": ("💼", "#f97316"),
        "NEWS_REQUEST": ("📰", "#3b82f6"),
        "EDUCATION": ("🎓", "#a855f7"),
        "GREETING": ("👋", "#22c55e"),
        "GENERAL": ("💬", "#64748b"),
        "ERROR": ("❌", "#ef4444"),
    }
    
    
    # ============================================================================
    # PAGE CONFIG & CUSTOM CSS
    # ============================================================================
    
    st.set_page_config(
        page_title="FinSight — Financial Assistant",
        page_icon="📊",
        layout="wide",
        initial_sidebar_state="expanded",
    )
    
    st.markdown("""
    <style>
        /* ---- Import Google Font ---- */
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    
        /* ---- Global Overrides ---- */
        html, body, [class*="css"] {
            font-family: 'Inter', sans-serif;
        }
    
        /* ---- Hide default Streamlit elements ---- */
        #MainMenu {visibility: hidden;}
        footer {visibility: hidden;}
        header {visibility: hidden;}
    
        /* ---- Main container ---- */
        .main .block-container {
            padding-top: 1.5rem;
            padding-bottom: 1rem;
            max-width: 900px;
        }
    
        /* ---- Chat input styling ---- */
        .stChatInput > div {
            border-radius: 16px !important;
            border: 1px solid rgba(99, 102, 241, 0.3) !important;
            background: rgba(15, 23, 42, 0.6) !important;
            backdrop-filter: blur(12px);
        }
        .stChatInput > div:focus-within {
            border-color: rgba(99, 102, 241, 0.7) !important;
            box-shadow: 0 0 20px rgba(99, 102, 241, 0.15) !important;
        }
    
        /* ---- Chat messages ---- */
        .stChatMessage {
            border-radius: 16px !important;
            margin-bottom: 0.5rem !important;
            border: 1px solid rgba(255,255,255,0.04) !important;
        }
    
        /* ---- Sidebar styling ---- */
        section[data-testid="stSidebar"] {
            background: linear-gradient(180deg, #0f172a 0%, #1e1b4b 100%);
            border-right: 1px solid rgba(99, 102, 241, 0.15);
        }
        section[data-testid="stSidebar"] .block-container {
            padding-top: 1rem;
        }
    
        /* ---- Hero header ---- */
        .hero-header {
            text-align: center;
            padding: 2rem 1rem 1rem;
            margin-bottom: 1rem;
        }
        .hero-header h1 {
            background: linear-gradient(135deg, #6366f1 0%, #a855f7 50%, #ec4899 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            font-size: 2.4rem;
            font-weight: 700;
            margin-bottom: 0.25rem;
            letter-spacing: -0.5px;
        }
        .hero-header p {
            color: #94a3b8;
            font-size: 0.95rem;
            font-weight: 400;
        }
    
        /* ---- Sidebar logo area ---- */
        .sidebar-brand {
            text-align: center;
            padding: 1.2rem 0.5rem;
            margin-bottom: 1rem;
            border-bottom: 1px solid rgba(99, 102, 241, 0.15);
        }
        .sidebar-brand h2 {
            background: linear-gradient(135deg, #6366f1, #a855f7);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            font-size: 1.5rem;
            margin-bottom: 0.15rem;
            font-weight: 700;
        }
        .sidebar-brand p {
            color: #64748b;
            font-size: 0.75rem;
            margin: 0;
        }
    
        /* ---- Section headers in sidebar ---- */
        .sidebar-section {
            color: #94a3b8;
            font-size: 0.7rem;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 1.2px;
            padding: 0.6rem 0 0.3rem;
            margin-top: 0.5rem;
        }
    
        /* ---- Intent badge ---- */
        .intent-badge {
            display: inline-block;
            padding: 2px 10px;
            border-radius: 12px;
            font-size: 0.68rem;
            font-weight: 600;
            letter-spacing: 0.5px;
            margin-top: 4px;
            opacity: 0.85;
        }
    
        /* ---- Suggestion chips ---- */
        .suggestion-container {
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            margin-top: 8px;
        }
    
        /* ---- Status indicator ---- */
        .status-dot {
            display: inline-block;
            width: 8px;
            height: 8px;
            border-radius: 50%;
            margin-right: 6px;
            animation: pulse-glow 2s infinite;
        }
        @keyframes pulse-glow {
            0%, 100% { opacity: 1; box-shadow: 0 0 4px currentColor; }
            50% { opacity: 0.5; box-shadow: 0 0 8px currentColor; }
        }
    
        /* ---- Welcome cards ---- */
        .welcome-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 10px;
            margin-top: 1rem;
        }
        .welcome-card {
            background: rgba(99, 102, 241, 0.06);
            border: 1px solid rgba(99, 102, 241, 0.12);
            border-radius: 12px;
            padding: 14px 16px;
            transition: all 0.2s ease;
        }
        .welcome-card:hover {
            border-color: rgba(99, 102, 241, 0.3);
            background: rgba(99, 102, 241, 0.1);
        }
        .welcome-card .card-icon {
            font-size: 1.4rem;
            margin-bottom: 6px;
        }
        .welcome-card .card-title {
            color: #e2e8f0;
            font-size: 0.82rem;
            font-weight: 600;
            margin-bottom: 3px;
        }
        .welcome-card .card-desc {
            color: #64748b;
            font-size: 0.72rem;
            line-height: 1.4;
        }
    
        /* ---- Divider ---- */
        .subtle-divider {
            border: none;
            border-top: 1px solid rgba(99, 102, 241, 0.1);
            margin: 0.8rem 0;
        }
    
        /* ---- Streamlit button overrides ---- */
        .stButton > button {
            border-radius: 10px !important;
            border: 1px solid rgba(99, 102, 241, 0.25) !important;
            background: rgba(99, 102, 241, 0.08) !important;
            color: #c7d2fe !important;
            font-size: 0.8rem !important;
            font-weight: 500 !important;
            padding: 0.35rem 0.9rem !important;
            transition: all 0.2s ease !important;
            width: 100%;
        }
        .stButton > button:hover {
            background: rgba(99, 102, 241, 0.2) !important;
            border-color: rgba(99, 102, 241, 0.5) !important;
            color: #fff !important;
        }
    </style>
    """, unsafe_allow_html=True)
    
    
    # ============================================================================
    # HELPER FUNCTIONS
    # ============================================================================
    
    def check_backend_health():
        """Check if the FastAPI backend is reachable."""
        try:
            r = requests.get(f"{API_BASE_URL}/health", timeout=3)
            return r.status_code == 200
        except Exception:
            return False
    
    
    def send_message(message: str, session_id: str = None) -> dict:
        """Send a message to the chatbot API and return the response."""
        payload = {"message": message}
        if session_id:
            payload["session_id"] = session_id
            
        r = requests.post(
            f"{API_BASE_URL}/chat",
            json=payload,
            timeout=30,
        )
        r.raise_for_status()
        return r.json()
    
    
    def render_intent_badge(intent: str):
        """Render a colored intent badge."""
        if not intent:
            return ""
        emoji, color = INTENT_BADGES.get(intent, ("💬", "#64748b"))
        return f'<span class="intent-badge" style="background: {color}22; color: {color}; border: 1px solid {color}44;">{emoji} {intent}</span>'
    
    
    # ============================================================================
    # SESSION STATE INITIALIZATION
    # ============================================================================
    
    if "messages" not in st.session_state:
        st.session_state.messages = []
    
    if "session_id" not in st.session_state:
        st.session_state.session_id = None
    
    if "backend_online" not in st.session_state:
        st.session_state.backend_online = check_backend_health()
    
    if "pending_suggestion" not in st.session_state:
        st.session_state.pending_suggestion = None
    
    
    # ============================================================================
    # SIDEBAR
    # ============================================================================
    
    with st.sidebar:
        # Brand
        st.markdown("""
        <div class="sidebar-brand">
            <h2>📊 FinSight</h2>
            <p>Intelligent Trading Companion</p>
        </div>
        """, unsafe_allow_html=True)
    
        # Status
        is_online = st.session_state.backend_online
        status_color = "#22c55e" if is_online else "#ef4444"
        status_text = "Backend Online" if is_online else "Backend Offline"
        st.markdown(
            f'<div style="display:flex;align-items:center;padding:0.3rem 0;margin-bottom:0.5rem;">'
            f'<span class="status-dot" style="color:{status_color};background:{status_color};"></span>'
            f'<span style="color:{status_color};font-size:0.78rem;font-weight:500;">{status_text}</span>'
            f'</div>',
            unsafe_allow_html=True,
        )
    
        if not is_online:
            if st.button("🔄 Retry Connection"):
                st.session_state.backend_online = check_backend_health()
                st.rerun()
    
        # Quick Actions
        st.markdown('<div class="sidebar-section">⚡ Quick Actions</div>', unsafe_allow_html=True)
    
        col1, col2 = st.columns(2)
        with col1:
            if st.button("📊 Market", key="qa_market"):
                st.session_state.pending_suggestion = "How is the market today?"
                st.rerun()
        with col2:
            if st.button("📰 News", key="qa_news"):
                st.session_state.pending_suggestion = "Show me latest market news"
                st.rerun()
    
        col3, col4 = st.columns(2)
        with col3:
            if st.button("🔍 Screener", key="qa_screener"):
                st.session_state.pending_suggestion = "Show me available stock screens"
                st.rerun()
        with col4:
            if st.button("📈 NIFTY", key="qa_nifty"):
                st.session_state.pending_suggestion = "What is Nifty 50 at right now?"
                st.rerun()
    
        st.markdown('<hr class="subtle-divider">', unsafe_allow_html=True)
    
        # Popular Stocks
        st.markdown('<div class="sidebar-section">🔥 Popular Stocks</div>', unsafe_allow_html=True)
    
        popular_stocks = ["RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK"]
        cols = st.columns(len(popular_stocks))
        for i, stock in enumerate(popular_stocks):
            with cols[i]:
                if st.button(stock, key=f"stock_{stock}"):
                    st.session_state.pending_suggestion = f"What is the price of {stock}?"
                    st.rerun()
    
        st.markdown('<hr class="subtle-divider">', unsafe_allow_html=True)
    
        # Sample Questions
        st.markdown('<div class="sidebar-section">💡 Try Asking</div>', unsafe_allow_html=True)
    
        sample_questions = [
            "Why is the market going down?",
            "Compare TCS and Infosys",
            "Explain stop-loss orders",
            "Analyze Reliance stock",
            "Last 5 day movement of TCS",
            "Show me undervalued stocks",
        ]
        for q in sample_questions:
            if st.button(f"→ {q}", key=f"sample_{q}"):
                st.session_state.pending_suggestion = q
                st.rerun()
    
        st.markdown('<hr class="subtle-divider">', unsafe_allow_html=True)
    
        # Session Info
        st.markdown('<div class="sidebar-section">🔧 Session</div>', unsafe_allow_html=True)
    
        sid_display = st.session_state.session_id
        if sid_display:
            st.markdown(
                f'<p style="color:#475569;font-size:0.7rem;word-break:break-all;">ID: {sid_display}</p>',
                unsafe_allow_html=True,
            )
    
        if st.button("🗑️ Clear Chat", key="clear_chat"):
            st.session_state.messages = []
            st.session_state.session_id = None
            st.rerun()
    
    
    # ============================================================================
    # MAIN CHAT AREA
    # ============================================================================
    
    # Hero header (only when chat is empty)
    if not st.session_state.messages:
        st.markdown("""
        <div class="hero-header">
            <h1>FinSight</h1>
            <p>Your AI-Powered Financial Assistant — Ask me anything about stocks, markets & trading</p>
        </div>
        """, unsafe_allow_html=True)
    
        # Welcome cards
        st.markdown("""
        <div class="welcome-grid">
            <div class="welcome-card">
                <div class="card-icon">💰</div>
                <div class="card-title">Stock Prices</div>
                <div class="card-desc">Get real-time prices for any NSE/BSE stock</div>
            </div>
            <div class="welcome-card">
                <div class="card-icon">📊</div>
                <div class="card-title">Market Trends</div>
                <div class="card-desc">NIFTY, SENSEX & sector-wise performance</div>
            </div>
            <div class="welcome-card">
                <div class="card-icon">📰</div>
                <div class="card-title">Financial News</div>
                <div class="card-desc">Latest headlines affecting your investments</div>
            </div>
            <div class="welcome-card">
                <div class="card-icon">🔍</div>
                <div class="card-title">Stock Screener</div>
                <div class="card-desc">Find undervalued, momentum & oversold stocks</div>
            </div>
        </div>
        """, unsafe_allow_html=True)
    
        st.markdown("", unsafe_allow_html=True)
    
    
    # Render existing chat messages
    for i, msg in enumerate(st.session_state.messages):
        with st.chat_message(msg["role"], avatar="👤" if msg["role"] == "user" else "📊"):
            st.markdown(msg["content"])
    
            # Show intent badge and suggestions for assistant messages
            if msg["role"] == "assistant":
                if msg.get("intent"):
                    badge_html = render_intent_badge(msg["intent"])
                    st.markdown(badge_html, unsafe_allow_html=True)
                    
                if msg.get("elapsed"):
                    st.caption(f"⚡ {msg['elapsed']}s")
    
            # Render suggestion chips for the LAST assistant message only
            if (
                msg["role"] == "assistant"
                and msg.get("suggestions")
                and i == len(st.session_state.messages) - 1
            ):
                suggestions = msg["suggestions"]
                cols = st.columns(min(len(suggestions), 4))
                for j, sugg in enumerate(suggestions):
                    with cols[j % 4]:
                        if st.button(f"💡 {sugg}", key=f"sugg_{i}_{j}"):
                            st.session_state.pending_suggestion = sugg
                            st.rerun()
    
    
    # ============================================================================
    # HANDLE INPUT (from chat box or suggestion click)
    # ============================================================================
    
    # Check for pending suggestion
    user_input = None
    if st.session_state.pending_suggestion:
        user_input = st.session_state.pending_suggestion
        st.session_state.pending_suggestion = None
    
    # Quick-question chips ABOVE input
    st.markdown("**Try asking:**")
    chip_cols = st.columns(4)
    chips = [("📊","Nifty today"), ("💰","Reliance price"),
             ("🔍","Undervalued stocks"), ("📚","Explain RSI")]
    for i, (icon, q) in enumerate(chips):
        if chip_cols[i].button(f"{icon} {q}", key=f"chip_{i}", use_container_width=True):
            st.session_state["prefill"] = q
            st.rerun()
    
    prefill = st.session_state.pop("prefill", None)
    chat_input = st.chat_input("Ask about stocks, markets, trading...") or prefill
    
    if chat_input:
        user_input = chat_input
    
    # Process the input
    if user_input:
        # Add user message
        st.session_state.messages.append({"role": "user", "content": user_input})
    
        with st.chat_message("user", avatar="👤"):
            st.markdown(user_input)
    
        # Get bot response — streaming
        with st.chat_message("assistant", avatar="📊"):
            message_placeholder = st.empty()
            status_placeholder = st.empty()
            full_response = ""
            metadata = {}
            start_time = time.time()
    
            try:
                with requests.post(
                    f"{API_BASE_URL}/stream",
                    json={"message": user_input,
                          "session_id": st.session_state.session_id},
                    stream=True,
                    timeout=60,
                ) as r:
                    if r.status_code == 429:
                        st.warning("Rate limit reached. Please wait before asking again.")
                        st.stop()
                    elif r.status_code != 200:
                        st.error(f"Backend error {r.status_code}. Check server logs.")
                        st.stop()
    
                    for line in r.iter_lines(decode_unicode=True):
                        if not line or not line.startswith("data: "):
                            continue
                        try:
                            data = json.loads(line[6:])
                        except json.JSONDecodeError:
                            continue
    
                        if data["type"] == "token":
                            full_response += data["content"]
                            message_placeholder.markdown(full_response + "▌")
                        elif data["type"] == "status":
                            status_placeholder.caption(data["content"])
                        elif data["type"] == "done":
                            metadata = data
    
            except requests.exceptions.ConnectionError:
                st.error("Cannot connect to backend. Run: `uvicorn main:app --reload`")
                st.stop()
            except requests.exceptions.Timeout:
                st.error("Request timed out. Try again.")
                st.stop()
            except requests.exceptions.ChunkedEncodingError:
                # Server dropped connection (e.g. hot-reload) — show what we received so far
                if not full_response:
                    st.error("Connection lost. Please try again.")
                    st.stop()
    
            elapsed = round(time.time() - start_time, 1)
    
            # Clear status indicator and streaming cursor
            status_placeholder.empty()
            message_placeholder.markdown(full_response)
    
            intent = metadata.get("intent")
            suggestions = metadata.get("suggestions", [])
            session_id = metadata.get("session_id")
    
            # Update session ID
            if session_id:
                st.session_state.session_id = session_id
    
            # Intent badge
            if intent:
                badge_html = render_intent_badge(intent)
                st.markdown(badge_html, unsafe_allow_html=True)
    
            st.caption(f"⚡ {elapsed}s")
    
            # Store the message
            st.session_state.messages.append({
                "role": "assistant",
                "content": full_response,
                "intent": intent,
                "suggestions": suggestions,
                "elapsed": elapsed
            })
    
            # Render suggestion chips
            if suggestions:
                cols = st.columns(min(len(suggestions), 4))
                for j, sugg in enumerate(suggestions):
                    with cols[j % 4]:
                        if st.button(f"💡 {sugg}", key=f"new_sugg_{j}"):
                            st.session_state.pending_suggestion = sugg
                            st.rerun()
    
    print('[OK] streamlit_app.py loaded successfully')
except ImportError as e:
    print(f'[SKIP] streamlit_app.py - Missing package: {e}')
    print(f'       Install with: pip install streamlit')

---
# 13. Stock Screener Engine

Full technical + fundamental analysis engine with pre-built screens (undervalued, momentum, oversold, etc.)

## 13.1 Technical Indicators (`screener/technical_indicators.py`)

Calculates RSI, SMA, EMA, MACD, Bollinger Bands, Supertrend, ADX, VWAP, Stochastic, OBV, and more.

Uses both custom calculations and the **pandas-ta** library for 130+ indicators.

In [ ]:
"""
Technical Indicators Module.
Calculates RSI, MACD, SMA, EMA, Bollinger Bands, and volume analysis
from historical price data fetched via yfinance.
"""

import logging
from typing import Optional, Dict, Any
from dataclasses import dataclass, field

logger = logging.getLogger(__name__)

try:
    import yfinance as yf
    import pandas as pd
    import numpy as np
except ImportError:
    yf = None
    pd = None
    np = None
    logger.warning("yfinance/pandas/numpy not available")

try:
    import pandas_ta as ta
    TA_AVAILABLE = True
except ImportError:
    TA_AVAILABLE = False
    logger.warning("pandas-ta not available — extended technical indicators disabled")


@dataclass
class IndicatorResult:
    """Result of a single indicator calculation."""
    value: Optional[float] = None
    signal: str = "NEUTRAL"  # "BUY", "SELL", "NEUTRAL"
    description: str = ""


class TechnicalIndicators:
    """
    Calculate technical analysis indicators from stock price data.
    Uses pandas for efficient vectorized calculations.
    """
    
    def __init__(self):
        """Initialize with empty state."""
        self._cache: Dict[str, Any] = {}
    
    def get_all_indicators(self, symbol: str, period: str = "6mo") -> Dict[str, Any]:
        """
        Calculate ALL technical indicators for a stock.
        
        Args:
            symbol: yfinance-compatible symbol (e.g., 'TCS.NS', 'NVDA')
            period: Data period for calculations (default: 6 months)
            
        Returns:
            Dict with all indicator values and signals
        """
        if pd is None or yf is None:
            logger.warning("pandas/yfinance not available")
            return self._empty_indicators()
        
        try:
            ticker = yf.Ticker(symbol)
            df = ticker.history(period=period)
            
            if df.empty or len(df) < 20:
                logger.warning(f"Insufficient data for {symbol}: {len(df)} rows")
                return self._empty_indicators()
            
            # Calculate all indicators
            indicators = {
                "rsi": self.calculate_rsi(df),
                "sma_20": self.calculate_sma(df, 20),
                "sma_50": self.calculate_sma(df, 50),
                "sma_200": self.calculate_sma(df, 200),
                "ema_12": self.calculate_ema(df, 12),
                "ema_26": self.calculate_ema(df, 26),
                "macd": self.calculate_macd(df),
                "bollinger": self.calculate_bollinger(df),
                "volume_analysis": self.calculate_volume_analysis(df),
                "current_price": float(df['Close'].iloc[-1]),
            }
            
            # Merge extended indicators if available
            extended = self.get_extended_indicators(df)
            indicators.update(extended)
            
            # Generate composite signal
            indicators["composite_signal"] = self._composite_signal(indicators)
            indicators["composite_score"] = self._composite_score(indicators)
            
            return indicators
            
        except Exception as e:
            logger.error(f"Error calculating indicators for {symbol}: {e}")
            return self._empty_indicators()
    
    # ========================================================================
    # RSI (Relative Strength Index)
    # ========================================================================
    
    def calculate_rsi(self, df, period: int = 14) -> Dict[str, Any]:
        """
        Calculate RSI (Relative Strength Index).
        
        RSI = 100 - (100 / (1 + RS))
        RS = Average Gain / Average Loss over N periods
        
        Signal:
            < 30: Oversold (potential BUY)
            > 70: Overbought (potential SELL)
            30-70: Neutral
        """
        try:
            delta = df['Close'].diff()
            
            gain = delta.where(delta > 0, 0.0)
            loss = (-delta).where(delta < 0, 0.0)
            
            avg_gain = gain.rolling(window=period, min_periods=period).mean()
            avg_loss = loss.rolling(window=period, min_periods=period).mean()
            
            rs = avg_gain / avg_loss.replace(0, float('inf'))
            rsi = 100 - (100 / (1 + rs))
            
            current_rsi = float(rsi.iloc[-1]) if not pd.isna(rsi.iloc[-1]) else None
            
            # Determine signal
            signal = "NEUTRAL"
            description = ""
            if current_rsi is not None:
                if current_rsi < 30:
                    signal = "BUY"
                    description = f"RSI {current_rsi:.1f} — Oversold (potential reversal up)"
                elif current_rsi > 70:
                    signal = "SELL"
                    description = f"RSI {current_rsi:.1f} — Overbought (potential reversal down)"
                else:
                    description = f"RSI {current_rsi:.1f} — Neutral zone"
            
            return {
                "value": round(current_rsi, 2) if current_rsi else None,
                "signal": signal,
                "description": description,
            }
        except Exception as e:
            logger.error(f"RSI calculation error: {e}")
            return {"value": None, "signal": "NEUTRAL", "description": "RSI unavailable"}
    
    # ========================================================================
    # SMA (Simple Moving Average)
    # ========================================================================
    
    def calculate_sma(self, df, period: int = 20) -> Dict[str, Any]:
        """
        Calculate Simple Moving Average.
        
        Signal:
            Price > SMA: Bullish
            Price < SMA: Bearish
        """
        try:
            if len(df) < period:
                return {"value": None, "signal": "NEUTRAL", "description": f"Insufficient data for SMA-{period}"}
            
            sma = df['Close'].rolling(window=period).mean()
            current_sma = float(sma.iloc[-1])
            current_price = float(df['Close'].iloc[-1])
            
            signal = "BUY" if current_price > current_sma else "SELL"
            position = "above" if current_price > current_sma else "below"
            diff_pct = ((current_price - current_sma) / current_sma) * 100
            
            return {
                "value": round(current_sma, 2),
                "signal": signal,
                "description": f"Price {position} SMA-{period} by {abs(diff_pct):.1f}%",
            }
        except Exception as e:
            logger.error(f"SMA-{period} calculation error: {e}")
            return {"value": None, "signal": "NEUTRAL", "description": f"SMA-{period} unavailable"}
    
    # ========================================================================
    # EMA (Exponential Moving Average)
    # ========================================================================
    
    def calculate_ema(self, df, period: int = 12) -> Dict[str, Any]:
        """
        Calculate Exponential Moving Average.
        EMA gives more weight to recent prices (reacts faster than SMA).
        """
        try:
            ema = df['Close'].ewm(span=period, adjust=False).mean()
            current_ema = float(ema.iloc[-1])
            current_price = float(df['Close'].iloc[-1])
            
            signal = "BUY" if current_price > current_ema else "SELL"
            position = "above" if current_price > current_ema else "below"
            
            return {
                "value": round(current_ema, 2),
                "signal": signal,
                "description": f"Price {position} EMA-{period}",
            }
        except Exception as e:
            logger.error(f"EMA-{period} calculation error: {e}")
            return {"value": None, "signal": "NEUTRAL", "description": f"EMA-{period} unavailable"}
    
    # ========================================================================
    # MACD (Moving Average Convergence Divergence)
    # ========================================================================
    
    def calculate_macd(self, df) -> Dict[str, Any]:
        """
        Calculate MACD.
        
        MACD Line = EMA(12) - EMA(26)
        Signal Line = EMA(9) of MACD Line
        Histogram = MACD - Signal
        
        Signal:
            MACD > Signal (histogram > 0): Bullish
            MACD < Signal (histogram < 0): Bearish
            Crossover: Strong signal
        """
        try:
            ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
            ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
            
            macd_line = ema_12 - ema_26
            signal_line = macd_line.ewm(span=9, adjust=False).mean()
            histogram = macd_line - signal_line
            
            current_macd = float(macd_line.iloc[-1])
            current_signal = float(signal_line.iloc[-1])
            current_histogram = float(histogram.iloc[-1])
            
            # Check for crossover (last 2 bars)
            prev_histogram = float(histogram.iloc[-2]) if len(histogram) >= 2 else 0
            
            signal = "NEUTRAL"
            description = ""
            
            if current_histogram > 0 and prev_histogram <= 0:
                signal = "BUY"
                description = "MACD bullish crossover (strong buy signal)"
            elif current_histogram < 0 and prev_histogram >= 0:
                signal = "SELL"
                description = "MACD bearish crossover (strong sell signal)"
            elif current_histogram > 0:
                signal = "BUY"
                description = f"MACD bullish (histogram: +{current_histogram:.2f})"
            else:
                signal = "SELL"
                description = f"MACD bearish (histogram: {current_histogram:.2f})"
            
            return {
                "macd": round(current_macd, 2),
                "signal_line": round(current_signal, 2),
                "histogram": round(current_histogram, 2),
                "signal": signal,
                "description": description,
            }
        except Exception as e:
            logger.error(f"MACD calculation error: {e}")
            return {"macd": None, "signal_line": None, "histogram": None, "signal": "NEUTRAL", "description": "MACD unavailable"}
    
    # ========================================================================
    # Bollinger Bands
    # ========================================================================
    
    def calculate_bollinger(self, df, period: int = 20, std_dev: float = 2.0) -> Dict[str, Any]:
        """
        Calculate Bollinger Bands.
        
        Upper Band = SMA(20) + 2 × StdDev
        Lower Band = SMA(20) - 2 × StdDev
        
        Signal:
            Price near lower band: Potential buy (oversold)
            Price near upper band: Potential sell (overbought)
            Band squeeze: Volatility breakout coming
        """
        try:
            sma = df['Close'].rolling(window=period).mean()
            std = df['Close'].rolling(window=period).std()
            
            upper = sma + (std_dev * std)
            lower = sma - (std_dev * std)
            
            current_price = float(df['Close'].iloc[-1])
            current_upper = float(upper.iloc[-1])
            current_lower = float(lower.iloc[-1])
            current_sma = float(sma.iloc[-1])
            
            # Band width (measure of volatility)
            band_width = ((current_upper - current_lower) / current_sma) * 100
            
            # Position within bands (0 = lower, 1 = upper)
            band_position = (current_price - current_lower) / (current_upper - current_lower) if (current_upper - current_lower) > 0 else 0.5
            
            signal = "NEUTRAL"
            description = ""
            
            if band_position <= 0.1:
                signal = "BUY"
                description = f"Price at lower Bollinger Band — oversold"
            elif band_position >= 0.9:
                signal = "SELL"
                description = f"Price at upper Bollinger Band — overbought"
            elif band_width < 5:
                description = f"Bollinger squeeze — volatility breakout expected"
            else:
                description = f"Price within Bollinger Bands ({band_position:.0%} position)"
            
            return {
                "upper": round(current_upper, 2),
                "lower": round(current_lower, 2),
                "middle": round(current_sma, 2),
                "band_width": round(band_width, 2),
                "band_position": round(band_position, 2),
                "signal": signal,
                "description": description,
            }
        except Exception as e:
            logger.error(f"Bollinger calculation error: {e}")
            return {"upper": None, "lower": None, "middle": None, "signal": "NEUTRAL", "description": "Bollinger unavailable"}
    
    # ========================================================================
    # Volume Analysis
    # ========================================================================
    
    def calculate_volume_analysis(self, df, period: int = 20) -> Dict[str, Any]:
        """
        Analyze volume relative to average.
        
        Volume Ratio = Current Volume / Avg Volume (20-day)
        
        Signal:
            > 2.0: Very high volume (confirmation of trend)
            > 1.5: Above average
            < 0.5: Very low volume (weak move)
        """
        try:
            avg_volume = df['Volume'].rolling(window=period).mean()
            current_volume = float(df['Volume'].iloc[-1])
            current_avg = float(avg_volume.iloc[-1])
            
            ratio = current_volume / current_avg if current_avg > 0 else 1.0
            
            description = ""
            if ratio > 2.0:
                description = f"Very high volume ({ratio:.1f}x avg) — strong conviction"
            elif ratio > 1.5:
                description = f"Above average volume ({ratio:.1f}x avg)"
            elif ratio < 0.5:
                description = f"Very low volume ({ratio:.1f}x avg) — weak move"
            else:
                description = f"Normal volume ({ratio:.1f}x avg)"
            
            return {
                "current_volume": int(current_volume),
                "avg_volume_20d": int(current_avg),
                "volume_ratio": round(ratio, 2),
                "description": description,
            }
        except Exception as e:
            logger.error(f"Volume analysis error: {e}")
            return {"current_volume": None, "avg_volume_20d": None, "volume_ratio": None, "description": "Volume unavailable"}
    
    # ========================================================================
    # Composite Signal & Score
    # ========================================================================
    
    def _composite_signal(self, indicators: Dict) -> str:
        """Generate a composite BUY/SELL/HOLD signal from all indicators."""
        buy_count = 0
        sell_count = 0
        total = 0
        
        # Count signals from each indicator
        for key in ["rsi", "sma_20", "sma_50", "ema_12", "ema_26"]:
            if key in indicators and indicators[key].get("signal"):
                total += 1
                if indicators[key]["signal"] == "BUY":
                    buy_count += 1
                elif indicators[key]["signal"] == "SELL":
                    sell_count += 1
        
        # MACD
        if "macd" in indicators and indicators["macd"].get("signal"):
            total += 1
            if indicators["macd"]["signal"] == "BUY":
                buy_count += 1
            elif indicators["macd"]["signal"] == "SELL":
                sell_count += 1
        
        # Bollinger
        if "bollinger" in indicators and indicators["bollinger"].get("signal"):
            total += 1
            if indicators["bollinger"]["signal"] == "BUY":
                buy_count += 1
            elif indicators["bollinger"]["signal"] == "SELL":
                sell_count += 1
        
        if total == 0:
            return "NEUTRAL"
        
        buy_pct = buy_count / total
        sell_pct = sell_count / total
        
        if buy_pct >= 0.6:
            return "BUY"
        elif sell_pct >= 0.6:
            return "SELL"
        else:
            return "HOLD"
    
    def _composite_score(self, indicators: Dict) -> float:
        """
        Generate a 0-100 composite technical score.
        
        50 = neutral, >70 = bullish, <30 = bearish
        """
        scores = []
        
        # RSI contributes (inverted scale for oversold)
        rsi_val = indicators.get("rsi", {}).get("value")
        if rsi_val is not None:
            # RSI 30 = score 80 (oversold = buy opportunity)
            # RSI 50 = score 50 (neutral)
            # RSI 70 = score 20 (overbought = sell risk)
            rsi_score = max(0, min(100, 100 - rsi_val))
            scores.append(rsi_score)
        
        # SMA signals
        for key in ["sma_20", "sma_50"]:
            sig = indicators.get(key, {}).get("signal")
            if sig == "BUY":
                scores.append(70)
            elif sig == "SELL":
                scores.append(30)
            else:
                scores.append(50)
        
        # MACD
        macd_sig = indicators.get("macd", {}).get("signal")
        if macd_sig == "BUY":
            scores.append(75)
        elif macd_sig == "SELL":
            scores.append(25)
        else:
            scores.append(50)
        
        # Bollinger position
        bb_pos = indicators.get("bollinger", {}).get("band_position")
        if bb_pos is not None:
            # Lower position = higher score (buy opportunity)
            bb_score = max(0, min(100, (1 - bb_pos) * 100))
            scores.append(bb_score)
        
        if not scores:
            return 50.0
        
        return round(sum(scores) / len(scores), 1)
    
    def _empty_indicators(self) -> Dict[str, Any]:
        """Return empty indicators when data is unavailable."""
        empty = {"value": None, "signal": "NEUTRAL", "description": "Data unavailable"}
        return {
            "rsi": empty.copy(),
            "sma_20": empty.copy(),
            "sma_50": empty.copy(),
            "sma_200": empty.copy(),
            "ema_12": empty.copy(),
            "ema_26": empty.copy(),
            "macd": {"macd": None, "signal_line": None, "histogram": None, "signal": "NEUTRAL", "description": "Data unavailable"},
            "bollinger": {"upper": None, "lower": None, "middle": None, "signal": "NEUTRAL", "description": "Data unavailable"},
            "volume_analysis": {"current_volume": None, "avg_volume_20d": None, "volume_ratio": None, "description": "Data unavailable"},
            "current_price": None,
            "composite_signal": "NEUTRAL",
            "composite_score": 50.0,
            "supertrend": {"value": None, "signal": "NEUTRAL", "description": "Data unavailable"},
            "adx": {"value": None, "signal": "NEUTRAL", "description": "Data unavailable"},
            "stochastic": {"k": None, "d": None, "signal": "NEUTRAL", "description": "Data unavailable"},
            "vwap": {"value": None, "signal": "NEUTRAL", "description": "Data unavailable"},
        }
    
    # ========================================================================
    # Extended Indicators (pandas-ta required)
    # ========================================================================
    
    def get_extended_indicators(self, df) -> Dict[str, Any]:
        """Calculate advanced indicators using pandas-ta."""
        extended = {}
        if not TA_AVAILABLE or len(df) < 50:
            return extended
            
        try:
            # 1. Supertrend (7, 3)
            sti = df.ta.supertrend(length=7, multiplier=3)
            if sti is not None and not sti.empty:
                direction_col = [c for c in sti.columns if c.startswith('SUPERTd_')][0]
                value_col = [c for c in sti.columns if c.startswith('SUPERT_')][0]
                
                direction = int(sti[direction_col].iloc[-1])
                st_value = float(sti[value_col].iloc[-1])
                
                signal = "BUY" if direction > 0 else "SELL"
                desc = f"Price above Supertrend ({st_value:.2f})" if direction > 0 else f"Price below Supertrend ({st_value:.2f})"
                
                extended["supertrend"] = {
                    "value": round(st_value, 2),
                    "signal": signal,
                    "description": desc
                }
                
            # 2. ADX (Average Directional Index) — trend strength
            adx_df = df.ta.adx(length=14)
            if adx_df is not None and not adx_df.empty:
                adx_col = [c for c in adx_df.columns if c.startswith('ADX_')][0]
                adx_val = float(adx_df[adx_col].iloc[-1])
                
                desc = "Strong Trend" if adx_val > 25 else "Weak/No Trend"
                
                extended["adx"] = {
                    "value": round(adx_val, 2),
                    "signal": "NEUTRAL", # ADX just shows strength, not direction
                    "description": f"ADX is {adx_val:.1f} ({desc})"
                }
                
            # 3. Stochastic Oscillator
            stoch_df = df.ta.stoch()
            if stoch_df is not None and not stoch_df.empty:
                k_col = [c for c in stoch_df.columns if c.startswith('STOCHk_')][0]
                d_col = [c for c in stoch_df.columns if c.startswith('STOCHd_')][0]
                
                k = float(stoch_df[k_col].iloc[-1])
                d = float(stoch_df[d_col].iloc[-1])
                
                signal = "NEUTRAL"
                desc = f"Stoch%K: {k:.1f}, %D: {d:.1f}"
                
                if k < 20 and d < 20:
                    signal = "BUY"
                    desc += " (Oversold)"
                elif k > 80 and d > 80:
                    signal = "SELL"
                    desc += " (Overbought)"
                    
                extended["stochastic"] = {
                    "k": round(k, 2),
                    "d": round(d, 2),
                    "signal": signal,
                    "description": desc
                }
                
            # 4. VWAP (Volume Weighted Average Price)
            # VWAP requires an intraday timeframe typically, but pandas-ta can approximate
            vwap_df = df.ta.vwap()
            if vwap_df is not None and not vwap_df.empty:
                vwap_val = float(vwap_df.iloc[-1])
                current_price = float(df['Close'].iloc[-1])
                
                signal = "BUY" if current_price > vwap_val else "SELL"
                pos = "above" if current_price > vwap_val else "below"
                
                extended["vwap"] = {
                    "value": round(vwap_val, 2),
                    "signal": signal,
                    "description": f"Price {pos} VWAP ({vwap_val:.2f})"
                }
                
        except Exception as e:
            logger.error(f"Error calculating extended indicators: {e}")
            
        return extended


# Singleton
_indicator_service = None

def get_indicator_service() -> TechnicalIndicators:
    """Get or create the technical indicators singleton."""
    global _indicator_service
    if _indicator_service is None:
        _indicator_service = TechnicalIndicators()
    return _indicator_service


## 13.2 Stock Screener (`screener/screener.py`)

Pre-built screens with composite scoring: BUY/SELL/HOLD signals based on technical + fundamental data.

In [ ]:
"""
Stock Screener Service.
Screens and ranks stocks using technical + fundamental analysis.
Uses MarketDataService (from chatbot's modules/) for data access.
"""

import asyncio
import logging
from typing import Optional, List, Dict, Any
from datetime import datetime

try:
    import yfinance as yf
except ImportError:
    yf = None

from common.models.schemas import (
    Market, StockAnalysis, TechnicalIndicators,
    FundamentalData, ScreenerResult
)
from common.data_services.market_data import get_market_data_service
from screener.technical_indicators import get_indicator_service

logger = logging.getLogger(__name__)


# ============================================================================
# NIFTY 50 STOCK UNIVERSE (Default scan list)
# ============================================================================

NIFTY_50 = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "KOTAKBANK",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "BAJFINANCE",
    "HCLTECH", "TITAN", "SUNPHARMA", "WIPRO", "ULTRACEMCO",
    "NESTLEIND", "TATAMOTORS", "TATASTEEL", "POWERGRID", "NTPC",
    "ADANIENT", "ADANIPORTS", "BAJAJFINSV", "TECHM", "ONGC",
    "COALINDIA", "JSWSTEEL", "DIVISLAB", "DRREDDY", "CIPLA",
    "EICHERMOT", "HEROMOTOCO", "BPCL", "GRASIM", "APOLLOHOSP",
    "BRITANNIA", "INDUSINDBK", "SBILIFE", "HDFCLIFE", "TATACONSUM",
    "M&M", "BAJAJ-AUTO", "UPL", "HINDALCO", "LTIM",
]


# Pre-built screen definitions
PREBUILT_SCREENS = {
    "undervalued": {
        "name": "Undervalued Stocks",
        "description": "Stocks with low PE, low PB, and good ROE — potentially undervalued by the market",
        "filters": {
            "pe_ratio_max": 20,
            "pb_ratio_max": 3,
            "roe_min": 12,
        }
    },
    "momentum": {
        "name": "Momentum Stocks",
        "description": "Stocks showing bullish momentum — RSI in sweet zone, above SMA50, positive MACD",
        "filters": {
            "rsi_min": 50,
            "rsi_max": 70,
            "above_sma_50": True,
            "macd_bullish": True,
        }
    },
    "oversold": {
        "name": "Oversold Stocks",
        "description": "Stocks with RSI below 30 or near lower Bollinger Band — potential bounce candidates",
        "filters": {
            "rsi_max": 35,
        }
    },
    "high_dividend": {
        "name": "High Dividend Yield",
        "description": "Stocks with dividend yield above 2% and reasonable PE",
        "filters": {
            "dividend_yield_min": 2.0,
            "pe_ratio_max": 25,
        }
    },
    "strong_fundamentals": {
        "name": "Strong Fundamentals",
        "description": "Stocks with high ROE, low debt, and good profit margins",
        "filters": {
            "roe_min": 15,
            "debt_to_equity_max": 1.0,
            "profit_margin_min": 10,
        }
    },
}


class ScreenerService:
    """
    Stock screener that combines technical and fundamental analysis.
    Uses the chatbot's MarketDataService for data and TechnicalIndicators for calculations.
    """
    
    def __init__(self):
        """Initialize screener with shared services."""
        self.market_service = get_market_data_service()
        self.indicators = get_indicator_service()
    
    # ========================================================================
    # Main Screening Methods
    # ========================================================================
    
    async def analyze_stock(self, symbol: str) -> Optional[StockAnalysis]:
        """
        Full analysis of a single stock (technical + fundamental).
        
        Args:
            symbol: Stock symbol (e.g., 'TCS', 'NVDA')
            
        Returns:
            StockAnalysis with all indicators and signal
        """
        try:
            # Detect market and get yfinance symbol
            yf_symbol, detected_market = self.market_service._detect_market(symbol)
            
            # Get fundamental data from yfinance
            fundamental = await self._get_fundamentals(yf_symbol, symbol)
            
            # Get technical indicators (needs price history)
            tech_data = self.indicators.get_all_indicators(yf_symbol, period="6mo")
            
            # Build TechnicalIndicators model
            technical = TechnicalIndicators(
                rsi=tech_data.get("rsi", {}).get("value"),
                sma_20=tech_data.get("sma_20", {}).get("value"),
                sma_50=tech_data.get("sma_50", {}).get("value"),
                sma_200=tech_data.get("sma_200", {}).get("value"),
                ema_12=tech_data.get("ema_12", {}).get("value"),
                ema_26=tech_data.get("ema_26", {}).get("value"),
                macd=tech_data.get("macd", {}).get("macd"),
                macd_signal=tech_data.get("macd", {}).get("signal_line"),
                macd_histogram=tech_data.get("macd", {}).get("histogram"),
                bollinger_upper=tech_data.get("bollinger", {}).get("upper"),
                bollinger_lower=tech_data.get("bollinger", {}).get("lower"),
                bollinger_position=tech_data.get("bollinger", {}).get("band_position"),
                volume_ratio=tech_data.get("volume_analysis", {}).get("volume_ratio"),
                supertrend_signal=tech_data.get("supertrend", {}).get("signal"),
                adx=tech_data.get("adx", {}).get("value"),
                vwap_position=tech_data.get("vwap", {}).get("signal"),
                stochastic_k=tech_data.get("stochastic", {}).get("k"),
            )
            
            # Get current price
            current_price = tech_data.get("current_price", 0)
            if current_price == 0:
                stock_price = await self.market_service.get_stock_price(symbol)
                if stock_price:
                    current_price = stock_price.price
            
            # Get price change
            stock_price = await self.market_service.get_stock_price(symbol)
            change_pct = stock_price.change_percent if stock_price else 0
            stock_name = stock_price.name if stock_price else symbol
            
            # Determine market enum
            market_enum = Market.US if detected_market == "US" else Market.NSE
            
            return StockAnalysis(
                symbol=symbol.upper(),
                name=stock_name,
                price=current_price,
                change_percent=change_pct,
                technical=technical,
                fundamental=fundamental,
                signal=tech_data.get("composite_signal", "NEUTRAL"),
                score=tech_data.get("composite_score", 50.0),
                market=market_enum,
            )
            
        except Exception as e:
            logger.error(f"Error analyzing {symbol}: {e}")
            return None
    
    async def screen_stocks(
        self,
        filters: Dict[str, Any],
        stock_list: List[str] = None,
        screen_name: str = "Custom Screen",
        description: str = "",
    ) -> ScreenerResult:
        """
        Screen a list of stocks using given filters.
        
        Args:
            filters: Dict of filter criteria
            stock_list: Stocks to screen (default: Nifty 50)
            screen_name: Name for the screen
            description: Description of the screen
            
        Returns:
            ScreenerResult with matching stocks sorted by score
        """
        stocks_to_scan = stock_list or NIFTY_50
        result_stocks = []
        
        # Analyze stocks concurrently in batches of 5
        batch_size = 5
        for i in range(0, len(stocks_to_scan), batch_size):
            batch = stocks_to_scan[i:i + batch_size]
            tasks = [self.analyze_stock(symbol) for symbol in batch]
            analyses = await asyncio.gather(*tasks, return_exceptions=True)
            
            for analysis in analyses:
                if isinstance(analysis, Exception):
                    logger.error(f"Analysis error: {analysis}")
                    continue
                if analysis is None:
                    continue
                
                # Apply filters
                if self._passes_filters(analysis, filters):
                    result_stocks.append(analysis)
        
        # Sort by composite score (highest first)
        result_stocks.sort(key=lambda s: s.score, reverse=True)
        
        return ScreenerResult(
            screen_name=screen_name,
            description=description,
            stocks=result_stocks,
            total_scanned=len(stocks_to_scan),
        )
    
    async def get_prebuilt_screen(self, screen_name: str) -> Optional[ScreenerResult]:
        """
        Run a pre-built screen by name.
        
        Args:
            screen_name: One of: undervalued, momentum, oversold, high_dividend, strong_fundamentals
            
        Returns:
            ScreenerResult or None if screen not found
        """
        screen = PREBUILT_SCREENS.get(screen_name.lower())
        if not screen:
            logger.warning(f"Unknown screen: {screen_name}")
            return None
        
        return await self.screen_stocks(
            filters=screen["filters"],
            screen_name=screen["name"],
            description=screen["description"],
        )
    
    def get_available_screens(self) -> List[Dict[str, str]]:
        """Get list of available pre-built screens."""
        return [
            {"id": key, "name": val["name"], "description": val["description"]}
            for key, val in PREBUILT_SCREENS.items()
        ]
    
    # ========================================================================
    # Helpers
    # ========================================================================
    
    async def _get_fundamentals(self, yf_symbol: str, symbol: str) -> FundamentalData:
        """Fetch fundamental data from yfinance."""
        try:
            if yf is None:
                return FundamentalData()
            
            ticker = yf.Ticker(yf_symbol)
            info = ticker.info
            
            # Calculate ROE: Net Income / Shareholder Equity
            net_income = info.get('netIncomeToCommon')
            equity = info.get('totalStockholderEquity')
            roe = (net_income / equity * 100) if (net_income and equity and equity != 0) else None
            
            eg = info.get('earningsGrowth')
            rg = info.get('revenueGrowth')
            inst = info.get('heldPercentInstitutions')
            
            return FundamentalData(
                pe_ratio=info.get('trailingPE'),
                pb_ratio=info.get('priceToBook'),
                roe=round(roe, 2) if roe else None,
                debt_to_equity=info.get('debtToEquity', None),
                eps=info.get('trailingEps'),
                dividend_yield=round(info.get('dividendYield', 0) * 100, 2) if info.get('dividendYield') else None,
                market_cap=info.get('marketCap'),
                revenue_growth=round(rg*100,1) if rg is not None else None,
                earnings_growth=round(eg*100,1) if eg is not None else None,
                current_ratio=info.get('currentRatio'),
                quick_ratio=info.get('quickRatio'),
                free_cash_flow=info.get('freeCashflow'),
                institutional_holding=round(inst*100,1) if inst is not None else None,
                peg_ratio=info.get('trailingPegRatio', info.get('pegRatio')),
                profit_margin=round(info.get('profitMargins', 0) * 100, 2) if info.get('profitMargins') else None,
                sector=info.get('sector'),
                industry=info.get('industry'),
            )
            
        except Exception as e:
            logger.error(f"Error fetching fundamentals for {symbol}: {e}")
            return FundamentalData()
    
    def _passes_filters(self, analysis: StockAnalysis, filters: Dict[str, Any]) -> bool:
        """Check if a stock analysis passes all given filters."""
        tech = analysis.technical
        fund = analysis.fundamental
        
        # PE ratio filters
        if "pe_ratio_max" in filters:
            if fund.pe_ratio is None or fund.pe_ratio > filters["pe_ratio_max"]:
                return False
        if "pe_ratio_min" in filters:
            if fund.pe_ratio is None or fund.pe_ratio < filters["pe_ratio_min"]:
                return False
        
        # PB ratio filters
        if "pb_ratio_max" in filters:
            if fund.pb_ratio is None or fund.pb_ratio > filters["pb_ratio_max"]:
                return False
        
        # ROE filter
        if "roe_min" in filters:
            if fund.roe is None or fund.roe < filters["roe_min"]:
                return False
        
        # Debt-to-Equity
        if "debt_to_equity_max" in filters:
            if fund.debt_to_equity is not None and fund.debt_to_equity > filters["debt_to_equity_max"]:
                return False
        
        # Dividend yield
        if "dividend_yield_min" in filters:
            if fund.dividend_yield is None or fund.dividend_yield < filters["dividend_yield_min"]:
                return False
        
        # Profit margin
        if "profit_margin_min" in filters:
            if fund.profit_margin is None or fund.profit_margin < filters["profit_margin_min"]:
                return False
        
        # RSI filters
        if "rsi_max" in filters:
            if tech.rsi is None or tech.rsi > filters["rsi_max"]:
                return False
        if "rsi_min" in filters:
            if tech.rsi is None or tech.rsi < filters["rsi_min"]:
                return False
        
        # Above SMA-50
        if filters.get("above_sma_50"):
            if tech.sma_50 is None or analysis.price <= tech.sma_50:
                return False
        
        # MACD bullish
        if filters.get("macd_bullish"):
            if tech.macd_histogram is None or tech.macd_histogram <= 0:
                return False
        
        # Score filter
        if "score_min" in filters:
            if analysis.score < filters["score_min"]:
                return False
        
        return True


# Singleton
_screener_service = None

def get_screener_service() -> ScreenerService:
    """Get or create the screener service singleton."""
    global _screener_service
    if _screener_service is None:
        _screener_service = ScreenerService()
    return _screener_service


## 13.3 Screener Formatter (`screener/screener_formatter.py`)

Formats analysis results into readable, emoji-rich output.

In [ ]:
"""
Screener Formatter.
Formats stock analysis and screener results into readable chat responses.
"""

from typing import Optional
from common.models.schemas import StockAnalysis, ScreenerResult, Market


class ScreenerFormatter:
    """Format screener results for chat display."""
    
    @staticmethod
    def format_analysis(analysis: StockAnalysis) -> str:
        """Format a single stock's full analysis."""
        currency = "$" if analysis.market == Market.US else "₹"
        signal_emoji = {
            "BUY": "🟢 BUY",
            "SELL": "🔴 SELL",
            "HOLD": "🟡 HOLD",
            "NEUTRAL": "⚪ NEUTRAL",
        }
        
        lines = []
        lines.append(f"📊 **{analysis.name or analysis.symbol}** ({analysis.symbol}) — Full Analysis")
        lines.append("")
        
        # Price & Signal
        lines.append(f"💰 **Price:** {currency}{analysis.price:,.2f}")
        chg = analysis.change_percent
        chg_icon = "🟢" if chg >= 0 else "🔴"
        lines.append(f"{chg_icon} **Change:** {chg:+.2f}%")
        lines.append(f"🎯 **Signal:** {signal_emoji.get(analysis.signal, analysis.signal)}")
        lines.append(f"📈 **Score:** {analysis.score:.0f}/100")
        lines.append("")
        
        # Technical Analysis
        tech = analysis.technical
        lines.append("**📉 Technical Analysis**")
        if tech.rsi is not None:
            rsi_tag = ""
            if tech.rsi < 30: rsi_tag = " (Oversold)"
            elif tech.rsi > 70: rsi_tag = " (Overbought)"
            lines.append(f"  • RSI(14): {tech.rsi:.1f}{rsi_tag}")
        if tech.sma_50 is not None:
            pos = "Above ✅" if analysis.price > tech.sma_50 else "Below ❌"
            lines.append(f"  • SMA-50: {currency}{tech.sma_50:,.2f} ({pos})")
        if tech.sma_200 is not None:
            pos = "Above ✅" if analysis.price > tech.sma_200 else "Below ❌"
            lines.append(f"  • SMA-200: {currency}{tech.sma_200:,.2f} ({pos})")
        if tech.macd is not None:
            macd_dir = "Bullish 📈" if (tech.macd_histogram or 0) > 0 else "Bearish 📉"
            lines.append(f"  • MACD: {tech.macd:.2f} ({macd_dir})")
        if tech.bollinger_position is not None:
            lines.append(f"  • Bollinger Position: {tech.bollinger_position:.0%}")
        if tech.supertrend_signal is not None:
            st_icon = "🟢" if tech.supertrend_signal == "BUY" else "🔴"
            lines.append(f"  • Supertrend: {st_icon} {tech.supertrend_signal}")
        if tech.vwap_position is not None:
            vwap_icon = "⬆️" if tech.vwap_position == "BUY" else "⬇️"
            lines.append(f"  • VWAP Pos: {vwap_icon} {tech.vwap_position}")
        if tech.stochastic_k is not None:
            lines.append(f"  • Stoch(%K): {tech.stochastic_k:.1f}")
        if tech.adx is not None:
            trend = "Strong" if tech.adx > 25 else "Weak"
            lines.append(f"  • ADX: {tech.adx:.1f} ({trend})")
        if tech.volume_ratio is not None:
            vol_tag = ""
            if tech.volume_ratio > 1.5: vol_tag = " (High)"
            elif tech.volume_ratio < 0.5: vol_tag = " (Low)"
            lines.append(f"  • Volume: {tech.volume_ratio:.1f}x avg{vol_tag}")
        lines.append("")
        
        # Fundamental Analysis
        fund = analysis.fundamental
        lines.append("**📋 Fundamental Analysis**")
        if fund.pe_ratio is not None:
            lines.append(f"  • P/E Ratio: {fund.pe_ratio:.1f}")
        if fund.pb_ratio is not None:
            lines.append(f"  • P/B Ratio: {fund.pb_ratio:.1f}")
        if fund.roe is not None:
            lines.append(f"  • ROE: {fund.roe:.1f}%")
        if fund.eps is not None:
            lines.append(f"  • EPS: {currency}{fund.eps:.2f}")
        if fund.debt_to_equity is not None:
            lines.append(f"  • Debt/Equity: {fund.debt_to_equity:.2f}")
        if fund.dividend_yield is not None:
            lines.append(f"  • Dividend Yield: {fund.dividend_yield:.1f}%")
        if fund.profit_margin is not None:
            lines.append(f"  • Profit Margin: {fund.profit_margin:.1f}%")
        if fund.revenue_growth is not None:
            lines.append(f"  • Revenue Growth: {fund.revenue_growth:+.1f}%")
        if fund.earnings_growth is not None:
            lines.append(f"  • Earnings Growth: {fund.earnings_growth:+.1f}%")
        if fund.institutional_holding is not None:
            lines.append(f"  • Inst. Holding: {fund.institutional_holding:.1f}%")
        if fund.current_ratio is not None:
            lines.append(f"  • Current Ratio: {fund.current_ratio:.2f}")
        if fund.quick_ratio is not None:
            lines.append(f"  • Quick Ratio: {fund.quick_ratio:.2f}")
        if fund.peg_ratio is not None:
            lines.append(f"  • PEG Ratio: {fund.peg_ratio:.2f}")
        if fund.market_cap is not None:
            lines.append(f"  • Market Cap: {ScreenerFormatter._format_market_cap(fund.market_cap, currency)}")
        if fund.free_cash_flow is not None:
            lines.append(f"  • Free Cash Flow: {currency}{fund.free_cash_flow/1e7:,.0f} Cr")
        if fund.sector:
            lines.append(f"  • Sector: {fund.sector}")
        lines.append("")
        lines.append("_Source: yfinance (technical + fundamental)_")
        
        return "\n".join(lines)
    
    @staticmethod
    def format_screener_result(result: ScreenerResult) -> str:
        """Format screener results as a table."""
        signal_emoji = {
            "BUY": "🟢",
            "SELL": "🔴",
            "HOLD": "🟡",
            "NEUTRAL": "⚪",
        }
        
        lines = []
        lines.append(f"📊 **{result.screen_name}**")
        if result.description:
            lines.append(f"_{result.description}_")
        lines.append("")
        lines.append(f"Scanned **{result.total_scanned}** stocks → **{len(result.stocks)}** matches")
        lines.append("")
        
        if not result.stocks:
            lines.append("No stocks matched the criteria. Try adjusting filters.")
            return "\n".join(lines)
        
        # Results table
        for i, stock in enumerate(result.stocks[:15], 1):  # Max 15 results
            sig = signal_emoji.get(stock.signal, "⚪")
            currency = "$" if stock.market == Market.US else "₹"
            
            pe_str = f"PE:{stock.fundamental.pe_ratio:.1f}" if stock.fundamental.pe_ratio else ""
            rsi_str = f"RSI:{stock.technical.rsi:.0f}" if stock.technical.rsi else ""
            
            chg = stock.change_percent
            chg_icon = "🟢" if chg >= 0 else "🔴"
            
            lines.append(
                f"**{i}.** {sig} **{stock.symbol}** — "
                f"{currency}{stock.price:,.2f} "
                f"{chg_icon}{chg:+.1f}% | "
                f"Score: {stock.score:.0f} | "
                f"{pe_str} {rsi_str}".strip()
            )
        
        if len(result.stocks) > 15:
            lines.append(f"\n_... and {len(result.stocks) - 15} more_")
        
        lines.append("")
        lines.append("_Source: Nifty 50 via yfinance_")
        
        return "\n".join(lines)
    
    @staticmethod
    def _format_market_cap(value: float, currency: str = "₹") -> str:
        """Format market cap in human-readable format."""
        if currency == "₹":
            if value >= 1e12:
                return f"₹{value / 1e12:.1f}L Cr"
            elif value >= 1e7:
                return f"₹{value / 1e7:.0f} Cr"
            else:
                return f"₹{value:,.0f}"
        else:
            if value >= 1e12:
                return f"${value / 1e12:.1f}T"
            elif value >= 1e9:
                return f"${value / 1e9:.1f}B"
            elif value >= 1e6:
                return f"${value / 1e6:.1f}M"
            else:
                return f"${value:,.0f}"


---
# 14. 🚀 Live Demo — Test the Chatbot

### Option A: Run the FastAPI server
```bash
# Terminal 1: Start the backend
cd project1
python main.py

# Terminal 2: Start the Streamlit UI
streamlit run streamlit_app.py
```

### Option B: Test individual components in this notebook

In [ ]:
# Quick test: Symbol resolution
# (This cell works standalone - no server needed)

# Uncomment to test:
# from chatbot.core.symbol_utils import resolve_symbol
# print(resolve_symbol('reliance'))    # -> RELIANCE
# print(resolve_symbol('reliace'))     # -> RELIANCE (fuzzy match via difflib!)
# print(resolve_symbol('tata moters')) # -> TATAMOTORS (fuzzy match via difflib!)
# print(resolve_symbol('nvidia'))      # -> NVDA
# print(resolve_symbol('HDFCBANK'))    # -> HDFCBANK

In [ ]:
# Quick test: Trading Knowledge Base

# Uncomment to test:
# from chatbot.modules.trading_assistant import get_trading_assistant
# assistant = get_trading_assistant()
# result = assistant.search('what is stop loss')
# if result:
#     print(assistant.format_response(result))

In [ ]:
# Quick test: Market Data (requires yfinance)

# Uncomment to test:
# import asyncio
# from common.data_services.market_data import get_market_data_service
# from chatbot.modules.market_formatter import MarketFormatter
#
# async def test_price():
#     service = get_market_data_service()
#     price = await service.get_stock_price('TCS')
#     if price:
#         print(MarketFormatter.format_stock_price(price))
#     else:
#         print('Could not fetch price')
#
# await test_price()  # In Jupyter, use await directly

In [ ]:
# Full agent test (requires GROQ_API_KEY in .env)

# Uncomment to test:
# from chatbot.agent import get_agent
#
# async def test_agent():
#     agent = get_agent()
#     await agent.initialize()
#     response = await agent.process_message('What is the price of TCS?')
#     print(f'Intent: {response.intent}')
#     print(f'Reply: {response.reply}')
#     print(f'Suggestions: {response.suggestions}')
#     await agent.shutdown()
#
# await test_agent()

---
# 📊 Project Summary

| Metric | Value |
|--------|-------|
| **Total Files** | 20+ Python files |
| **Architecture** | LangGraph ReAct Agent |
| **LLM** | Groq (LLaMA 3.1 70B) — free tier |
| **Data Sources** | yfinance, Screener.in, Google News RSS, Wikipedia |
| **Frontend** | Streamlit |
| **Backend** | FastAPI + Uvicorn |
| **Memory** | AsyncSqliteSaver (persistent) |
| **Deployment** | Railway |

---

**Built with ❤️ as a complete AI chatbot project for Indian stock markets.**